# MSc. Thesis, transfer learning in urban noise prediction

## Creating calculation points

#### Tampere

In [ ]:
import requests
import json
import pandas as pd
import geopandas as gpd
import io
import osmnx as ox
import shapely
import numpy as np
import rasterio
from shapely.geometry.point import Point
from shapely.geometry.linestring import LineString

# Functions from Santeri

def split_by_num_points(row, point_gdf):
    if not np.isnan(row.num_points) :
        points = int(row.num_points)
        if points > 1 and row.oneway == True:
            splitter = shapely.geometry.MultiPoint([row.geometry.interpolate((i/4), normalized=True) for i in range(1, points)])
            split_line = shapely.ops.split(shapely.ops.snap(row.geometry, splitter, 0.1), splitter)
            for segment in split_line:
                if point_gdf[point_gdf['tmsNumber']==row.tmsNumber].intersects(segment).any():
                    return segment
        else:
            return row.geometry

def road_is_not_link(row):
    if type(row['highway']) == list:
        for road_type in row['highway']:
            if 'link' in road_type:
                return False
    elif 'link' in row['highway']:
        return False
    return True

def capture_roads_by_names(row, roadnames):
    flattened = []
    for r in roadnames:
        if str(r) == 'nan':
            continue
        if type(r) == list:
            for sub_r in r:
                flattened.append(sub_r)
        else:
            flattened.append(r)
    return any(roadname in str(row['name']) for roadname in flattened)

def merge_multilines(row):
    if row['geometry'].type == "MultiLineString":
        single_line = shapely.ops.linemerge(row['geometry'])
        return single_line
    return row['geometry']

def add_z_coord_as_height_point(point):
    # z = get_elevation_at_lat_lon('mosaic.tif', point.x, point.y)
    # if z == None:
    #     z = 10
    z = 10
    
    return shapely.geometry.Point(point.x, point.y, z)

def combine_point_rows_to_one(row):
    base_cols = [f'points{i}' for i in range(1, 13)]
    extra = [f'points{i}' for i in range(13, 19)] if row.tmsNumber in [457, 458, 439] else []
    multipoints = [getattr(row, col) for col in base_cols + extra]

    points = []
    for multipoint in multipoints:
        for point in multipoint:
            points.append(add_z_coord_as_height_point(point))
    return shapely.geometry.MultiPoint(points)


def spread_points_on_lines(line, distance_delta):
    distances = np.arange(0, line.length, distance_delta)
    points = [line.interpolate(distance) for distance in distances] + [line.boundary[1]]
    multipoint = shapely.ops.unary_union(points)  # or new_line = LineString(points)
    return multipoint

def get_elevation_at_lat_lon(tif_path, lon, lat):
    with rasterio.open(tif_path) as dataset:
        
        # Convert lon/lat to pixel coordinates
        row, col = dataset.index(lon, lat)
        
        # Validate that the indices are within the dataset bounds
        if row < 0 or row >= dataset.height or col < 0 or col >= dataset.width:
            print("Warning: Calculated pixel coordinates are out of dataset bounds.")
            return np.nan  # or handle as desired
        
        # Read the elevation value from the defined window
        elevation = dataset.read(1, window=((row, row+1), (col, col+1)))
        
        # Check if elevation data was read successfully
        if elevation.size == 0:
            print("Warning: No elevation data found in the specified window.")
            return np.nan
        
        return elevation[0, 0]

def height_from_coords(x_coord, y_coord, bbox, heightmaps):
    #for x, y in zip(xs, ys):
    within_box = None
    for square, coords in bbox.items():
        x1,y1,x2,y2 = coords
        if x_coord > x1 and x_coord < x2 and y_coord > y1 and y_coord < y2:
            within_box = square
    if not within_box:
        return
    row, col = rasterio.transform.rowcol(heightmaps[within_box].transform, x_coord, y_coord)
    height_value = heightmaps[within_box].read(1)[row][col]
    #print(height_value)
    return height_value


def my_interpolate(input_line, input_dist, normalized=False):
    '''
    From: https://stackoverflow.com/a/69489292
    
    Function that interpolates the coordinates of a shapely LineString.
    Note: If you use this function on a MultiLineString geometry, it will 
    "flatten" the geometry and consider all the points in it to be 
    consecutively connected. For example, consider the following shape: 
        MultiLineString(((0,0),(0,2)),((0,4),(0,6)))
    In this case, this function will assume that there is no gap between
    (0,2) and (0,4). Instead, the function will assume that these points
    all connected. Explicitly, the MultiLineString above will be 
    interpreted instead as the following shape:
        LineString((0,0),(0,2),(0,4),(0,6))

    Parameters
    ----------
    input_line : shapely.geometry.Linestring or shapely.geometry.MultiLineString
        (Multi)LineString whose coordinates you want to interpolate
    input_dist : float
        Distance used to calculate the interpolation point
    normalized : boolean
        Flag that indicates whether or not the `input_dist` argument should be
        interpreted as being an absolute number or a percentage that is 
        relative to the total distance or not.
        When this flag is set to "False", the `input_dist` argument is assumed 
        to be an actual absolute distance from the starting point of the 
        geometry. When this flag is set to "True", the `input_dist` argument 
        is assumed to represent the relative distance with respect to the 
        geometry's full distance.
        The default is False.

    Returns
    -------
    shapely.geometry.Point
        The shapely geometry of the interpolated Point.

    '''
    # Making sure the entry value is a LineString or MultiLineString
    if ((input_line.type.lower() != 'linestring') and 
        (input_line.type.lower() != 'multilinestring')):
        return None
    
    # Extracting the coordinates from the geometry
    if input_line.type.lower()[:len('multi')] == 'multi':
        # In case it's a multilinestring, this step "flattens" the points
        coords = [item for sub_list in [list(this_geom.coords) for 
                                        this_geom in input_line.geoms] 
                  for item in sub_list]
    else:
        coords = [tuple(coord) for coord in list(input_line.coords)]
    
    # Transforming the list of coordinates into a numpy array for 
    # ease of manipulation
    coords = np.array(coords)
    
    # Calculating the distances between points
    dists = ((coords[:-1] - coords[1:])**2).sum(axis=1)**0.5
    
    # Calculating the cumulative distances
    dists_cum = np.append(0,dists.cumsum())
    
    # Finding the total distance
    dist_total = dists_cum[-1]
    
    # Finding appropriate use of the `input_dist` value
    if normalized == False:
        input_dist_abs = input_dist
        input_dist_rel = input_dist / dist_total
    else:
        input_dist_abs = input_dist * dist_total
        input_dist_rel = input_dist
    
    # Taking care of some edge cases
    if ((input_dist_rel < 0) or 
        (input_dist_rel > 1) or 
        (input_dist_abs < 0) or 
        (input_dist_abs > dist_total)):
        return None
    elif ((input_dist_rel == 0) or (input_dist_abs == 0)):
        return shapely.geometry.Point(coords[0])
    elif ((input_dist_rel == 1) or (input_dist_abs == dist_total)):
        return shapely.geometry.Point(coords[-1])
    
    # Finding which point is immediately before and after the input distance
    pt_before_idx = np.arange(dists_cum.shape[0])[(dists_cum <= input_dist_abs)].max()
    pt_after_idx  = np.arange(dists_cum.shape[0])[(dists_cum >= input_dist_abs)].min()
    
    pt_before = coords[pt_before_idx]
    pt_after = coords[pt_after_idx]
    seg_full_dist = dists[pt_before_idx]
    dist_left = input_dist_abs - dists_cum[pt_before_idx]
    
    # Calculating the interpolated coordinates
    interpolated_coords = ((dist_left / seg_full_dist) * (pt_after - pt_before)) + pt_before
    
    # Creating a shapely geometry
    interpolated_point = shapely.geometry.point.Point(interpolated_coords)
    
    return interpolated_point

# ----------------------------------
# 1. Load TMS Station Data
# ----------------------------------

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))

gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
gdf.crs = "EPSG:4326"

# Filter stations for Tampere area
tampere = pd.concat([
    gdf.loc[["tampere" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tre" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["rautaharkko" in c.lower() for c in list(gdf['name'])]]
])
tampere = tampere[
    (tampere.name != "vt7_Treksilä") # &
    # (tampere.name != "vt3_Tampere_Myllypuro")
]
tampere = tampere.to_crs(4326)
# Tampere = tampere[tampere['tmsNumber'] != 471]

# -------------------------------------
# 2. Download and Prepare Road Networks
# -------------------------------------

G = ox.graph_from_place(
    "Tampere, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G, strict=False)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
tunnels = gdf_edges[gdf_edges["tunnel"] == "yes"]

G = ox.graph_from_place(
    "Tampere, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
gdf_edges = gdf_edges[gdf_edges.apply(road_is_not_link, axis=1)]

tunnels = tunnels.reset_index(drop=True)
simple_edges = gdf_edges.reset_index(drop=True)
simple_edges['tunnel'] = 'null'

# --------------------------------
# 3. Match Roads with TMS Stations
# --------------------------------

tampere = tampere.to_crs(3067)
tampere['buffer'] = tampere['geometry'].buffer(5)
tampere = tampere.set_geometry('buffer')

new_gdf = gpd.sjoin(
    simple_edges[['name','oneway','geometry','highway','tunnel']],
    tampere[['buffer','tmsNumber']],
    predicate='intersects'
).reset_index(drop=True)

only_roads_with_stations = simple_edges[
    simple_edges.apply(capture_roads_by_names,
                       args=([i for i in new_gdf['name']], ), axis=1)
    | simple_edges['name'].isnull()
]
only_roads_with_stations_tunnels = gpd.overlay(
    only_roads_with_stations[['name','oneway','geometry','highway']],
    tunnels[['tunnel','geometry']], how='union'
)
only_roads_with_stations_tunnels = only_roads_with_stations_tunnels.join(
    gpd.sjoin(tampere, only_roads_with_stations_tunnels)
      .groupby("index_right").size().rename("num_points"),
    how="left",
)

joined_station_in_tunnel = gpd.sjoin(
    only_roads_with_stations_tunnels,
    tampere[['buffer','tmsNumber']],
    op='intersects'
)
joined_station_in_tunnel['geometry'] = joined_station_in_tunnel.apply(merge_multilines, axis=1)

new_gdf = new_gdf.set_index('index_right', drop=True)
new_gdf.index.name = None

new_gdf = new_gdf.join(
    gpd.sjoin(tampere, new_gdf).groupby("index_right").size().rename("num_points"),
    how="left",
)
new_gdf.geometry = new_gdf.apply(split_by_num_points, axis=1, args=(tampere,))

test_buffer = new_gdf.copy()
test_buffer['geometry'] = new_gdf.buffer(40, single_sided=False)
only_roads_with_stations_tunnels.index.name = None
test = gpd.clip(
    test_buffer,
    only_roads_with_stations_tunnels[only_roads_with_stations_tunnels['tunnel'].isnull()]
)
both_lanes = gpd.overlay(test, tunnels[tunnels['name'] == 'Tampereen itäinen kehätie'], how='union')

just_one_tunnel = joined_station_in_tunnel[joined_station_in_tunnel['tunnel'] == 'yes']
new_gdf = new_gdf.overlay(just_one_tunnel, how='identity')
new_gdf['tunnel'] = new_gdf.apply(
    lambda row: "yes" if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' else np.nan, axis=1
)
new_gdf = new_gdf.rename(columns={
    'num_points_1': 'num_points',
    'name_1': 'name',
    'oneway_1': 'oneway',
    'highway_1': 'highway',
    'tmsNumber_1': 'tmsNumber'
})
new_gdf = new_gdf.drop(['name_2','oneway_2','tunnel_1','tunnel_2','highway_2','tmsNumber_2'], axis=1)

# --------------------------------------------
# 4. Build the Single-Lane Road Representation
# --------------------------------------------

new_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(50, single_sided=False), crs='epsg:3067')
other_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(5, single_sided=False), crs='epsg:3067')
gdf_one_lane = both_lanes.clip(
    new_buffer.overlay(other_buffer, how='difference'), keep_geom_type=True
)

gdf_filter = gdf_one_lane.copy()
for i, row in gdf_one_lane.iterrows():
    if isinstance(row.geometry, shapely.geometry.collection.GeometryCollection):
        lines = [shape for shape in row.geometry if isinstance(shape, shapely.geometry.LineString)]
        merged = shapely.ops.linemerge(lines)
        gdf_filter.at[i, 'geometry'] = merged

tampere['buffer'] = tampere.buffer(165)
tampere = tampere.set_geometry('buffer')
gdf_filtered = gpd.sjoin(gdf_filter, tampere, op='intersects').set_geometry('geometry_left')
gdf_filtered.index.name = None
gdf_filtered['geometry'] = gdf_filtered['geometry_left']
gdf_filtered = gdf_filtered.set_geometry('geometry')

gdf_combined = new_gdf.append(gdf_filtered)
gdf_combined = gdf_combined[gdf_combined.geom_type != 'Point']

gdf_combined['tunnel'] = gdf_combined.apply(
    lambda row: "yes"
    if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' or row['tunnel'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['oneway'] = gdf_combined.apply(
    lambda row: "yes"
    if row['oneway_2'] == 'yes' or row['oneway_1'] == 'yes' or row['oneway'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['highway'] = gdf_combined.apply(
    lambda row: (
        row['highway_1'] if isinstance(row['highway_1'], (str, list))
        else (row['highway_2'] if isinstance(row['highway_2'], (str, list))
              else row['highway'])
    ),
    axis=1
)
gdf_combined['tmsNumber'] = gdf_combined.apply(
    lambda row: (
        row['tmsNumber']
        if not pd.isna(row['tmsNumber'])
        else (row['tmsNumber_left'] if not pd.isna(row['tmsNumber_left'])
              else row['tmsNumber_right'])
    ),
    axis=1
)
gdf_combined['name'] = gdf_combined.apply(
    lambda row: (
        row['name_1'] if isinstance(row['name_1'], str)
        else (row['name_2'] if isinstance(row['name_2'], str)
              else row['name'])
    ),
    axis=1
)
gdf_combined = gdf_combined[['name','oneway','geometry','highway','tunnel','tmsNumber']]
gdf_combined = gdf_combined.reset_index(drop=True)
gdf_combined = gdf_combined.explode()

# --------------------------------------
# 5. Offset Lines & Create Point Samples
# --------------------------------------

for dist_i, offset_dist in enumerate([10,15,21,28,37,48,62,78,98,120,146,174,204,240,280,320,370,420], start=1):
    gdf_combined[f'parallel{dist_i}'] = gdf_combined['geometry'].apply(
        lambda row: row.parallel_offset(offset_dist, side="right",
                                        resolution=16, join_style=2,
                                        mitre_limit=10)
    )
    gdf_combined[f'points{dist_i}'] = gdf_combined[f'parallel{dist_i}'].apply(
        spread_points_on_lines, args=(15,)
    )

gdf_combined['all_points'] = gdf_combined.apply(combine_point_rows_to_one, axis=1)
point_cloud = gdf_combined.set_geometry('all_points')

tmsNumbers = []
points = []
for _, row in point_cloud.iterrows():
    tmsNumber = row['tmsNumber']
    for point in row['all_points']:
        points.append(point)
        tmsNumbers.append(tmsNumber)

calculation_points = gpd.GeoDataFrame(
    {"tmsnumber": tmsNumbers, "geometry": points},
    geometry=points, crs="epsg:3067"
)

# -------------------------
# 6. Save Results
# -------------------------
calculation_points.to_file("new_data.geojson", driver="GeoJSON")

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
C:\Users\Dan\AppData\Local\Temp\ipykernel_23164\967227351.py:287: UserWarning: `keep_geom_type=True` in overlay resulted in 5 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  only_roads_with_stations_tunnels = gpd.overlay(
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\IPython\core\interactiveshell.py:3490: FutureWarning: The `op` parameter is deprecated and will be removed in a future rel

### oulu data

#### only few stations

In [7]:
import requests
import json
import pandas as pd
import geopandas as gpd
import io
import osmnx as ox
import shapely
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import split, snap, unary_union, linemerge

# ----------------------------
# 1. Load TMS Station Data
# ----------------------------
stations = requests.get("https://tie.digitraffic.fi/api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))
gdf = gpd.GeoDataFrame.from_features(stations_t["features"], crs="EPSG:4326")

# Filter to specific stations
selected_stations = ['st815_Oulunsalo'] # 'vt4_Oulu_Mäntylä', 'vt4_Oulu_Hiironen', 'st815_Oulunsalo'
oulu_stations = gdf[gdf['name'].isin(selected_stations)].to_crs(3067)

# ----------------------------
# 2. Prepare Road Network
# ----------------------------
def road_is_not_link(row):
    highway = row['highway']
    if isinstance(highway, list):
        return not any('link' in h for h in highway)
    return 'link' not in highway

# Download and process Oulu roads
G = ox.graph_from_place("Oulu, Finland", 
                       custom_filter='["highway"~"motorway|trunk|primary|secondary"]',
                       simplify=False)
G = ox.simplify_graph(G)
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.to_crs(3067).reset_index(drop=True)
edges = edges[edges.apply(road_is_not_link, axis=1)]

# ----------------------------
# 3. Match Roads with Stations
# ----------------------------
oulu_stations = oulu_stations.set_geometry(oulu_stations.buffer(50))  # Modified line

# Perform spatial join with proper geometry
roads_with_stations = gpd.sjoin(
    edges, 
    oulu_stations[['geometry', 'tmsNumber']],  # Now using proper geometry
    predicate='intersects'
).reset_index(drop=True)

# ----------------------------
# 4. Generate Offset Points
# ----------------------------
def spread_points(line, interval):
    if line.is_empty or line.length == 0:
        return []
    points = [line.interpolate(d) for d in np.arange(0, line.length, interval)]
    return points + [line.boundary[1]]

offsets = [10,15,21,28,37,48,62,78,98,120,146,174]
points = []
tms_numbers = []

for _, road in roads_with_stations.iterrows():
    geom = road.geometry
    for offset in offsets:
        # Generate points on both sides
        for side in ['left', 'right']:
            offset_line = geom.parallel_offset(offset, side, resolution=16, join_style=2, mitre_limit=10)
            if offset_line.is_empty:
                continue
            # Ensure correct orientation
            if side == 'left' and offset_line.geom_type == 'LineString':
                offset_line = LineString(offset_line.coords[::-1])
            pts = spread_points(offset_line, 15)
            points.extend(pts)
            tms_numbers.extend([road.tmsNumber] * len(pts))

# ----------------------------
# 5. Create Final GeoDataFrame
# ----------------------------
gdf_points = gpd.GeoDataFrame(
    {'tmsNumber': tms_numbers, 'geometry': points},
    crs=roads_with_stations.crs
)
gdf_points.to_file("stations_points2.geojson", driver="GeoJSON")

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


ValueError: Cannot write empty DataFrame to file.

In [ ]:
Vt4_Oulu_Kontinkangas_Flex
vt4_Oulu_Haukipudas
vt4_Oulu_Kello
Vt20_Oulu_Honkimaa_Flex

vt20_Rusko2
vt20_Kiiminki

Same:
vt4_Oulu_Intiö_LML


#### main

In [2]:
import requests
import json
import pandas as pd
import geopandas as gpd
import io
import osmnx as ox
import shapely
import numpy as np
import rasterio
from shapely.geometry.point import Point
from shapely.geometry.linestring import LineString

# Functions from Santeri

def split_by_num_points(row, point_gdf):
    if not np.isnan(row.num_points) :
        points = int(row.num_points)
        if points > 1 and row.oneway == True:
            splitter = shapely.geometry.MultiPoint([row.geometry.interpolate((i/4), normalized=True) for i in range(1, points)])
            split_line = shapely.ops.split(shapely.ops.snap(row.geometry, splitter, 0.1), splitter)
            for segment in split_line:
                if point_gdf[point_gdf['tmsNumber']==row.tmsNumber].intersects(segment).any():
                    return segment
        else:
            return row.geometry

def road_is_not_link(row):
    if type(row['highway']) == list:
        for road_type in row['highway']:
            if 'link' in road_type:
                return False
    elif 'link' in row['highway']:
        return False
    return True

def capture_roads_by_names(row, roadnames):
    flattened = []
    for r in roadnames:
        if str(r) == 'nan':
            continue
        if type(r) == list:
            for sub_r in r:
                flattened.append(sub_r)
        else:
            flattened.append(r)
    return any(roadname in str(row['name']) for roadname in flattened)

def merge_multilines(row):
    if row['geometry'].type == "MultiLineString":
        single_line = shapely.ops.linemerge(row['geometry'])
        return single_line
    return row['geometry']

def add_z_coord_as_height_point(point):
    # z = get_elevation_at_lat_lon('mosaic.tif', point.x, point.y)
    # if z == None:
    #     z = 10
    z = 10
    
    return shapely.geometry.Point(point.x, point.y, z)

def combine_point_rows_to_one(row):
    base_cols = [f'points{i}' for i in range(1, 13)]
    extra = [f'points{i}' for i in range(13, 19)] if row.tmsNumber in [1223, 1247, 1257, 1254] else []
    extra2 = [f'points{i}' for i in range(19, 22)] if row.tmsNumber in [1239, 1237, 1250, 21201] else []
    multipoints = [getattr(row, col) for col in base_cols + extra + extra2]

    points = []
    for multipoint in multipoints:
        for point in multipoint:
            points.append(add_z_coord_as_height_point(point))
    return shapely.geometry.MultiPoint(points)


def spread_points_on_lines(line, distance_delta):
    distances = np.arange(0, line.length, distance_delta)
    points = [line.interpolate(distance) for distance in distances] + [line.boundary[1]]
    multipoint = shapely.ops.unary_union(points)  # or new_line = LineString(points)
    return multipoint

def get_elevation_at_lat_lon(tif_path, lon, lat):
    with rasterio.open(tif_path) as dataset:
        
        # Convert lon/lat to pixel coordinates
        row, col = dataset.index(lon, lat)
        
        # Validate that the indices are within the dataset bounds
        if row < 0 or row >= dataset.height or col < 0 or col >= dataset.width:
            print("Warning: Calculated pixel coordinates are out of dataset bounds.")
            return np.nan  # or handle as desired
        
        # Read the elevation value from the defined window
        elevation = dataset.read(1, window=((row, row+1), (col, col+1)))
        
        # Check if elevation data was read successfully
        if elevation.size == 0:
            print("Warning: No elevation data found in the specified window.")
            return np.nan
        
        return elevation[0, 0]

def height_from_coords(x_coord, y_coord, bbox, heightmaps):
    #for x, y in zip(xs, ys):
    within_box = None
    for square, coords in bbox.items():
        x1,y1,x2,y2 = coords
        if x_coord > x1 and x_coord < x2 and y_coord > y1 and y_coord < y2:
            within_box = square
    if not within_box:
        return
    row, col = rasterio.transform.rowcol(heightmaps[within_box].transform, x_coord, y_coord)
    height_value = heightmaps[within_box].read(1)[row][col]
    #print(height_value)
    return height_value


def my_interpolate(input_line, input_dist, normalized=False):
    '''
    From: https://stackoverflow.com/a/69489292
    
    Function that interpolates the coordinates of a shapely LineString.
    Note: If you use this function on a MultiLineString geometry, it will 
    "flatten" the geometry and consider all the points in it to be 
    consecutively connected. For example, consider the following shape: 
        MultiLineString(((0,0),(0,2)),((0,4),(0,6)))
    In this case, this function will assume that there is no gap between
    (0,2) and (0,4). Instead, the function will assume that these points
    all connected. Explicitly, the MultiLineString above will be 
    interpreted instead as the following shape:
        LineString((0,0),(0,2),(0,4),(0,6))

    Parameters
    ----------
    input_line : shapely.geometry.Linestring or shapely.geometry.MultiLineString
        (Multi)LineString whose coordinates you want to interpolate
    input_dist : float
        Distance used to calculate the interpolation point
    normalized : boolean
        Flag that indicates whether or not the `input_dist` argument should be
        interpreted as being an absolute number or a percentage that is 
        relative to the total distance or not.
        When this flag is set to "False", the `input_dist` argument is assumed 
        to be an actual absolute distance from the starting point of the 
        geometry. When this flag is set to "True", the `input_dist` argument 
        is assumed to represent the relative distance with respect to the 
        geometry's full distance.
        The default is False.

    Returns
    -------
    shapely.geometry.Point
        The shapely geometry of the interpolated Point.

    '''
    # Making sure the entry value is a LineString or MultiLineString
    if ((input_line.type.lower() != 'linestring') and 
        (input_line.type.lower() != 'multilinestring')):
        return None
    
    # Extracting the coordinates from the geometry
    if input_line.type.lower()[:len('multi')] == 'multi':
        # In case it's a multilinestring, this step "flattens" the points
        coords = [item for sub_list in [list(this_geom.coords) for 
                                        this_geom in input_line.geoms] 
                  for item in sub_list]
    else:
        coords = [tuple(coord) for coord in list(input_line.coords)]
    
    # Transforming the list of coordinates into a numpy array for 
    # ease of manipulation
    coords = np.array(coords)
    
    # Calculating the distances between points
    dists = ((coords[:-1] - coords[1:])**2).sum(axis=1)**0.5
    
    # Calculating the cumulative distances
    dists_cum = np.append(0,dists.cumsum())
    
    # Finding the total distance
    dist_total = dists_cum[-1]
    
    # Finding appropriate use of the `input_dist` value
    if normalized == False:
        input_dist_abs = input_dist
        input_dist_rel = input_dist / dist_total
    else:
        input_dist_abs = input_dist * dist_total
        input_dist_rel = input_dist
    
    # Taking care of some edge cases
    if ((input_dist_rel < 0) or 
        (input_dist_rel > 1) or 
        (input_dist_abs < 0) or 
        (input_dist_abs > dist_total)):
        return None
    elif ((input_dist_rel == 0) or (input_dist_abs == 0)):
        return shapely.geometry.Point(coords[0])
    elif ((input_dist_rel == 1) or (input_dist_abs == dist_total)):
        return shapely.geometry.Point(coords[-1])
    
    # Finding which point is immediately before and after the input distance
    pt_before_idx = np.arange(dists_cum.shape[0])[(dists_cum <= input_dist_abs)].max()
    pt_after_idx  = np.arange(dists_cum.shape[0])[(dists_cum >= input_dist_abs)].min()
    
    pt_before = coords[pt_before_idx]
    pt_after = coords[pt_after_idx]
    seg_full_dist = dists[pt_before_idx]
    dist_left = input_dist_abs - dists_cum[pt_before_idx]
    
    # Calculating the interpolated coordinates
    interpolated_coords = ((dist_left / seg_full_dist) * (pt_after - pt_before)) + pt_before
    
    # Creating a shapely geometry
    interpolated_point = shapely.geometry.point.Point(interpolated_coords)
    
    return interpolated_point

# ----------------------------------
# 1. Load TMS Station Data
# ----------------------------------

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))

gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
gdf.crs = "EPSG:4326"

# Filter stations for Tampere area
oulu = pd.concat([
    gdf.loc[["oulu" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tobo testipiste" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["st815_lentokentäntie" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_rusko2" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_kiiminki" in c.lower() for c in list(gdf['name'])]]
])
# oulu_stations = oulu[oulu.name != "kt45_Oulunkylä"]

oulu_stations = oulu[
    (oulu.name != "vt7_Treksilä") &
    (oulu.name != "Vt4_Oulu_Kontinkangas_Flex") &
    (oulu.name != "vt4_Oulu_Haukipudas") &
    (oulu.name != "vt4_Oulu_Kello") &
    (oulu.name != "Vt20_Oulu_Honkimaa_Flex")
]

tampere = oulu_stations.to_crs(4326)
# Tampere = tampere[tampere['tmsNumber'] != 471]

# -------------------------------------
# 2. Download and Prepare Road Networks
# -------------------------------------

G = ox.graph_from_place(
    "Oulu, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G, strict=False)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
tunnels = gdf_edges[gdf_edges["tunnel"] == "yes"]

G = ox.graph_from_place(
    "Oulu, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
gdf_edges = gdf_edges[gdf_edges.apply(road_is_not_link, axis=1)]

tunnels = tunnels.reset_index(drop=True)
simple_edges = gdf_edges.reset_index(drop=True)
simple_edges['tunnel'] = 'null'

# --------------------------------
# 3. Match Roads with TMS Stations
# --------------------------------

tampere = tampere.to_crs(3067)
tampere['buffer'] = tampere['geometry'].buffer(5)
tampere = tampere.set_geometry('buffer')

new_gdf = gpd.sjoin(
    simple_edges[['name','oneway','geometry','highway','tunnel']],
    tampere[['buffer','tmsNumber']],
    predicate='intersects'
).reset_index(drop=True)

only_roads_with_stations = simple_edges[
    simple_edges.apply(capture_roads_by_names,
                       args=([i for i in new_gdf['name']], ), axis=1)
    | simple_edges['name'].isnull()
]
only_roads_with_stations_tunnels = gpd.overlay(
    only_roads_with_stations[['name','oneway','geometry','highway']],
    tunnels[['tunnel','geometry']], how='union'
)
only_roads_with_stations_tunnels = only_roads_with_stations_tunnels.join(
    gpd.sjoin(tampere, only_roads_with_stations_tunnels)
      .groupby("index_right").size().rename("num_points"),
    how="left",
)

joined_station_in_tunnel = gpd.sjoin(
    only_roads_with_stations_tunnels,
    tampere[['buffer','tmsNumber']],
    op='intersects'
)
joined_station_in_tunnel['geometry'] = joined_station_in_tunnel.apply(merge_multilines, axis=1)

new_gdf = new_gdf.set_index('index_right', drop=True)
new_gdf.index.name = None

new_gdf = new_gdf.join(
    gpd.sjoin(tampere, new_gdf).groupby("index_right").size().rename("num_points"),
    how="left",
)
new_gdf.geometry = new_gdf.apply(split_by_num_points, axis=1, args=(tampere,))

test_buffer = new_gdf.copy()
test_buffer['geometry'] = new_gdf.buffer(40, single_sided=False)
only_roads_with_stations_tunnels.index.name = None
test = gpd.clip(
    test_buffer,
    only_roads_with_stations_tunnels[only_roads_with_stations_tunnels['tunnel'].isnull()]
)
# both_lanes = gpd.overlay(test, tunnels[tunnels['name'] == 'Tampereen itäinen kehätie'], how='union')
both_lanes = gpd.overlay(test, tunnels, how='union')

just_one_tunnel = joined_station_in_tunnel[joined_station_in_tunnel['tunnel'] == 'yes']
new_gdf = new_gdf.overlay(just_one_tunnel, how='identity')
new_gdf['tunnel'] = new_gdf.apply(
    lambda row: "yes" if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' else np.nan, axis=1
)
new_gdf = new_gdf.rename(columns={
    'num_points_1': 'num_points',
    'name_1': 'name',
    'oneway_1': 'oneway',
    'highway_1': 'highway',
    'tmsNumber_1': 'tmsNumber'
})
new_gdf = new_gdf.drop(['name_2','oneway_2','tunnel_1','tunnel_2','highway_2','tmsNumber_2'], axis=1)

# --------------------------------------------
# 4. Build the Single-Lane Road Representation
# --------------------------------------------

new_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(50, single_sided=False), crs='epsg:3067')
other_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(5, single_sided=False), crs='epsg:3067')
gdf_one_lane = both_lanes.clip(
    new_buffer.overlay(other_buffer, how='difference'), keep_geom_type=True
)

gdf_filter = gdf_one_lane.copy()
for i, row in gdf_one_lane.iterrows():
    if isinstance(row.geometry, shapely.geometry.collection.GeometryCollection):
        lines = [shape for shape in row.geometry if isinstance(shape, shapely.geometry.LineString)]
        merged = shapely.ops.linemerge(lines)
        gdf_filter.at[i, 'geometry'] = merged

tampere['buffer'] = tampere.buffer(165)
tampere = tampere.set_geometry('buffer')
gdf_filtered = gpd.sjoin(gdf_filter, tampere, op='intersects').set_geometry('geometry_left')
gdf_filtered.index.name = None
gdf_filtered['geometry'] = gdf_filtered['geometry_left']
gdf_filtered = gdf_filtered.set_geometry('geometry')

gdf_combined = new_gdf.append(gdf_filtered)
gdf_combined = gdf_combined[gdf_combined.geom_type != 'Point']

gdf_combined['tunnel'] = gdf_combined.apply(
    lambda row: "yes"
    if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' or row['tunnel'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['oneway'] = gdf_combined.apply(
    lambda row: "yes"
    if row['oneway_2'] == 'yes' or row['oneway_1'] == 'yes' or row['oneway'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['highway'] = gdf_combined.apply(
    lambda row: (
        row['highway_1'] if isinstance(row['highway_1'], (str, list))
        else (row['highway_2'] if isinstance(row['highway_2'], (str, list))
              else row['highway'])
    ),
    axis=1
)
gdf_combined['tmsNumber'] = gdf_combined.apply(
    lambda row: (
        row['tmsNumber']
        if not pd.isna(row['tmsNumber'])
        else (row['tmsNumber_left'] if not pd.isna(row['tmsNumber_left'])
              else row['tmsNumber_right'])
    ),
    axis=1
)
gdf_combined['name'] = gdf_combined.apply(
    lambda row: (
        row['name_1'] if isinstance(row['name_1'], str)
        else (row['name_2'] if isinstance(row['name_2'], str)
              else row['name'])
    ),
    axis=1
)
gdf_combined = gdf_combined[['name','oneway','geometry','highway','tunnel','tmsNumber']]
gdf_combined = gdf_combined.reset_index(drop=True)
gdf_combined = gdf_combined.explode()

# --------------------------------------
# 5. Offset Lines & Create Point Samples
# --------------------------------------

for dist_i, offset_dist in enumerate([10,15,21,28,37,48,62,78,98,120,146,174,204,240,280,320,370,420,1,3,5], start=1):
    gdf_combined[f'parallel{dist_i}'] = gdf_combined['geometry'].apply(
        lambda row: row.parallel_offset(offset_dist, side="right",
                                        resolution=16, join_style=2,
                                        mitre_limit=10)
    )
    gdf_combined[f'points{dist_i}'] = gdf_combined[f'parallel{dist_i}'].apply(
        spread_points_on_lines, args=(15,)
    )

gdf_combined['all_points'] = gdf_combined.apply(combine_point_rows_to_one, axis=1)
point_cloud = gdf_combined.set_geometry('all_points')

tmsNumbers = []
points = []
for _, row in point_cloud.iterrows():
    tmsNumber = row['tmsNumber']
    for point in row['all_points']:
        points.append(point)
        tmsNumbers.append(tmsNumber)

calculation_points = gpd.GeoDataFrame(
    {"tmsnumber": tmsNumbers, "geometry": points},
    geometry=points, crs="epsg:3067"
)

# -------------------------
# 6. Save Results
# -------------------------
calculation_points.to_file("oulu_data2.geojson", driver="GeoJSON")

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
C:\Users\Dan\AppData\Local\Temp\ipykernel_5948\2395868930.py:296: UserWarning: `keep_geom_type=True` in overlay resulted in 4 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  only_roads_with_stations_tunnels = gpd.overlay(
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\IPython\core\interactiveshell.py:3490: FutureWarning: The `op` parameter is deprecated and will be removed in a future rel

## OSM roads

In [10]:
import requests
import json
import pandas as pd
import geopandas as gpd
import io
import osmnx as ox
import shapely
import numpy as np
import rasterio
from shapely.geometry.point import Point
from shapely.geometry.linestring import LineString

# Functions from Santeri

def split_by_num_points(row, point_gdf):
    if not np.isnan(row.num_points) :
        points = int(row.num_points)
        if points > 1 and row.oneway == True:
            splitter = shapely.geometry.MultiPoint([row.geometry.interpolate((i/4), normalized=True) for i in range(1, points)])
            split_line = shapely.ops.split(shapely.ops.snap(row.geometry, splitter, 0.1), splitter)
            for segment in split_line:
                if point_gdf[point_gdf['tmsNumber']==row.tmsNumber].intersects(segment).any():
                    return segment
        else:
            return row.geometry

def road_is_not_link(row):
    if type(row['highway']) == list:
        for road_type in row['highway']:
            if 'link' in road_type:
                return False
    elif 'link' in row['highway']:
        return False
    return True

def capture_roads_by_names(row, roadnames):
    flattened = []
    for r in roadnames:
        if str(r) == 'nan':
            continue
        if type(r) == list:
            for sub_r in r:
                flattened.append(sub_r)
        else:
            flattened.append(r)
    return any(roadname in str(row['name']) for roadname in flattened)

def merge_multilines(row):
    if row['geometry'].type == "MultiLineString":
        single_line = shapely.ops.linemerge(row['geometry'])
        return single_line
    return row['geometry']

def add_z_coord_as_height_point(point):
    # z = get_elevation_at_lat_lon('mosaic.tif', point.x, point.y)
    # if z == None:
    #     z = 10
    z = 10
    
    return shapely.geometry.Point(point.x, point.y, z)

def combine_point_rows_to_one(row):
    base_cols = [f'points{i}' for i in range(1, 13)]
    extra = [f'points{i}' for i in range(13, 19)] if row.tmsNumber in [457, 458, 439] else []
    multipoints = [getattr(row, col) for col in base_cols + extra]

    points = []
    for multipoint in multipoints:
        for point in multipoint:
            points.append(add_z_coord_as_height_point(point))
    return shapely.geometry.MultiPoint(points)


def spread_points_on_lines(line, distance_delta):
    distances = np.arange(0, line.length, distance_delta)
    points = [line.interpolate(distance) for distance in distances] + [line.boundary[1]]
    multipoint = shapely.ops.unary_union(points)  # or new_line = LineString(points)
    return multipoint

def get_elevation_at_lat_lon(tif_path, lon, lat):
    with rasterio.open(tif_path) as dataset:
        
        # Convert lon/lat to pixel coordinates
        row, col = dataset.index(lon, lat)
        
        # Validate that the indices are within the dataset bounds
        if row < 0 or row >= dataset.height or col < 0 or col >= dataset.width:
            print("Warning: Calculated pixel coordinates are out of dataset bounds.")
            return np.nan  # or handle as desired
        
        # Read the elevation value from the defined window
        elevation = dataset.read(1, window=((row, row+1), (col, col+1)))
        
        # Check if elevation data was read successfully
        if elevation.size == 0:
            print("Warning: No elevation data found in the specified window.")
            return np.nan
        
        return elevation[0, 0]

def height_from_coords(x_coord, y_coord, bbox, heightmaps):
    #for x, y in zip(xs, ys):
    within_box = None
    for square, coords in bbox.items():
        x1,y1,x2,y2 = coords
        if x_coord > x1 and x_coord < x2 and y_coord > y1 and y_coord < y2:
            within_box = square
    if not within_box:
        return
    row, col = rasterio.transform.rowcol(heightmaps[within_box].transform, x_coord, y_coord)
    height_value = heightmaps[within_box].read(1)[row][col]
    #print(height_value)
    return height_value


def my_interpolate(input_line, input_dist, normalized=False):
    '''
    From: https://stackoverflow.com/a/69489292
    
    Function that interpolates the coordinates of a shapely LineString.
    Note: If you use this function on a MultiLineString geometry, it will 
    "flatten" the geometry and consider all the points in it to be 
    consecutively connected. For example, consider the following shape: 
        MultiLineString(((0,0),(0,2)),((0,4),(0,6)))
    In this case, this function will assume that there is no gap between
    (0,2) and (0,4). Instead, the function will assume that these points
    all connected. Explicitly, the MultiLineString above will be 
    interpreted instead as the following shape:
        LineString((0,0),(0,2),(0,4),(0,6))

    Parameters
    ----------
    input_line : shapely.geometry.Linestring or shapely.geometry.MultiLineString
        (Multi)LineString whose coordinates you want to interpolate
    input_dist : float
        Distance used to calculate the interpolation point
    normalized : boolean
        Flag that indicates whether or not the `input_dist` argument should be
        interpreted as being an absolute number or a percentage that is 
        relative to the total distance or not.
        When this flag is set to "False", the `input_dist` argument is assumed 
        to be an actual absolute distance from the starting point of the 
        geometry. When this flag is set to "True", the `input_dist` argument 
        is assumed to represent the relative distance with respect to the 
        geometry's full distance.
        The default is False.

    Returns
    -------
    shapely.geometry.Point
        The shapely geometry of the interpolated Point.

    '''
    # Making sure the entry value is a LineString or MultiLineString
    if ((input_line.type.lower() != 'linestring') and 
        (input_line.type.lower() != 'multilinestring')):
        return None
    
    # Extracting the coordinates from the geometry
    if input_line.type.lower()[:len('multi')] == 'multi':
        # In case it's a multilinestring, this step "flattens" the points
        coords = [item for sub_list in [list(this_geom.coords) for 
                                        this_geom in input_line.geoms] 
                  for item in sub_list]
    else:
        coords = [tuple(coord) for coord in list(input_line.coords)]
    
    # Transforming the list of coordinates into a numpy array for 
    # ease of manipulation
    coords = np.array(coords)
    
    # Calculating the distances between points
    dists = ((coords[:-1] - coords[1:])**2).sum(axis=1)**0.5
    
    # Calculating the cumulative distances
    dists_cum = np.append(0,dists.cumsum())
    
    # Finding the total distance
    dist_total = dists_cum[-1]
    
    # Finding appropriate use of the `input_dist` value
    if normalized == False:
        input_dist_abs = input_dist
        input_dist_rel = input_dist / dist_total
    else:
        input_dist_abs = input_dist * dist_total
        input_dist_rel = input_dist
    
    # Taking care of some edge cases
    if ((input_dist_rel < 0) or 
        (input_dist_rel > 1) or 
        (input_dist_abs < 0) or 
        (input_dist_abs > dist_total)):
        return None
    elif ((input_dist_rel == 0) or (input_dist_abs == 0)):
        return shapely.geometry.Point(coords[0])
    elif ((input_dist_rel == 1) or (input_dist_abs == dist_total)):
        return shapely.geometry.Point(coords[-1])
    
    # Finding which point is immediately before and after the input distance
    pt_before_idx = np.arange(dists_cum.shape[0])[(dists_cum <= input_dist_abs)].max()
    pt_after_idx  = np.arange(dists_cum.shape[0])[(dists_cum >= input_dist_abs)].min()
    
    pt_before = coords[pt_before_idx]
    pt_after = coords[pt_after_idx]
    seg_full_dist = dists[pt_before_idx]
    dist_left = input_dist_abs - dists_cum[pt_before_idx]
    
    # Calculating the interpolated coordinates
    interpolated_coords = ((dist_left / seg_full_dist) * (pt_after - pt_before)) + pt_before
    
    # Creating a shapely geometry
    interpolated_point = shapely.geometry.point.Point(interpolated_coords)
    
    return interpolated_point

# ----------------------------------
# 1. Load TMS Station Data
# ----------------------------------

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))

gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
gdf.crs = "EPSG:4326"

# Filter stations for Tampere area
tampere = pd.concat([
    gdf.loc[["tampere" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tre" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["rautaharkko" in c.lower() for c in list(gdf['name'])]]
])
tampere = tampere[
    (tampere.name != "vt7_Treksilä") # &
    # (tampere.name != "vt3_Tampere_Myllypuro")
]
tampere = tampere.to_crs(4326)
# Tampere = tampere[tampere['tmsNumber'] != 471]

# -------------------------------------
# 2. Download and Prepare Road Networks
# -------------------------------------

G = ox.graph_from_place(
    "Tampere, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G, strict=False)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
tunnels = gdf_edges[gdf_edges["tunnel"] == "yes"]

G = ox.graph_from_place(
    "Tampere, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
gdf_edges = gdf_edges[gdf_edges.apply(road_is_not_link, axis=1)]

tunnels = tunnels.reset_index(drop=True)
simple_edges = gdf_edges.reset_index(drop=True)
simple_edges['tunnel'] = 'null'

# --------------------------------
# 3. Match Roads with TMS Stations
# --------------------------------

tampere = tampere.to_crs(3067)
tampere['buffer'] = tampere['geometry'].buffer(5)
tampere = tampere.set_geometry('buffer')

new_gdf = gpd.sjoin(
    simple_edges[['name','oneway','geometry','highway','tunnel']],
    tampere[['buffer','tmsNumber']],
    predicate='intersects'
).reset_index(drop=True)

only_roads_with_stations = simple_edges[
    simple_edges.apply(capture_roads_by_names,
                       args=([i for i in new_gdf['name']], ), axis=1)
    | simple_edges['name'].isnull()
]
only_roads_with_stations_tunnels = gpd.overlay(
    only_roads_with_stations[['name','oneway','geometry','highway']],
    tunnels[['tunnel','geometry']], how='union'
)
only_roads_with_stations_tunnels = only_roads_with_stations_tunnels.join(
    gpd.sjoin(tampere, only_roads_with_stations_tunnels)
      .groupby("index_right").size().rename("num_points"),
    how="left",
)

joined_station_in_tunnel = gpd.sjoin(
    only_roads_with_stations_tunnels,
    tampere[['buffer','tmsNumber']],
    op='intersects'
)
joined_station_in_tunnel['geometry'] = joined_station_in_tunnel.apply(merge_multilines, axis=1)

new_gdf = new_gdf.set_index('index_right', drop=True)
new_gdf.index.name = None

new_gdf = new_gdf.join(
    gpd.sjoin(tampere, new_gdf).groupby("index_right").size().rename("num_points"),
    how="left",
)
new_gdf.geometry = new_gdf.apply(split_by_num_points, axis=1, args=(tampere,))

test_buffer = new_gdf.copy()
test_buffer['geometry'] = new_gdf.buffer(40, single_sided=False)
only_roads_with_stations_tunnels.index.name = None
test = gpd.clip(
    test_buffer,
    only_roads_with_stations_tunnels[only_roads_with_stations_tunnels['tunnel'].isnull()]
)
both_lanes = gpd.overlay(test, tunnels[tunnels['name'] == 'Tampereen itäinen kehätie'], how='union')

just_one_tunnel = joined_station_in_tunnel[joined_station_in_tunnel['tunnel'] == 'yes']
new_gdf = new_gdf.overlay(just_one_tunnel, how='identity')
new_gdf['tunnel'] = new_gdf.apply(
    lambda row: "yes" if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' else np.nan, axis=1
)
new_gdf = new_gdf.rename(columns={
    'num_points_1': 'num_points',
    'name_1': 'name',
    'oneway_1': 'oneway',
    'highway_1': 'highway',
    'tmsNumber_1': 'tmsNumber'
})
new_gdf = new_gdf.drop(['name_2','oneway_2','tunnel_1','tunnel_2','highway_2','tmsNumber_2'], axis=1)

# --------------------------------------------
# 4. Build the Single-Lane Road Representation
# --------------------------------------------

new_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(50, single_sided=False), crs='epsg:3067')
other_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(5, single_sided=False), crs='epsg:3067')
gdf_one_lane = both_lanes.clip(
    new_buffer.overlay(other_buffer, how='difference'), keep_geom_type=True
)

gdf_filter = gdf_one_lane.copy()
for i, row in gdf_one_lane.iterrows():
    if isinstance(row.geometry, shapely.geometry.collection.GeometryCollection):
        lines = [shape for shape in row.geometry if isinstance(shape, shapely.geometry.LineString)]
        merged = shapely.ops.linemerge(lines)
        gdf_filter.at[i, 'geometry'] = merged

tampere['buffer'] = tampere.buffer(165)
tampere = tampere.set_geometry('buffer')
gdf_filtered = gpd.sjoin(gdf_filter, tampere, op='intersects').set_geometry('geometry_left')
gdf_filtered.index.name = None
gdf_filtered['geometry'] = gdf_filtered['geometry_left']
gdf_filtered = gdf_filtered.set_geometry('geometry')

gdf_combined = new_gdf.append(gdf_filtered)
gdf_combined = gdf_combined[gdf_combined.geom_type != 'Point']

gdf_combined['tunnel'] = gdf_combined.apply(
    lambda row: "yes"
    if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' or row['tunnel'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['oneway'] = gdf_combined.apply(
    lambda row: "yes"
    if row['oneway_2'] == 'yes' or row['oneway_1'] == 'yes' or row['oneway'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['highway'] = gdf_combined.apply(
    lambda row: (
        row['highway_1'] if isinstance(row['highway_1'], (str, list))
        else (row['highway_2'] if isinstance(row['highway_2'], (str, list))
              else row['highway'])
    ),
    axis=1
)
gdf_combined['tmsNumber'] = gdf_combined.apply(
    lambda row: (
        row['tmsNumber']
        if not pd.isna(row['tmsNumber'])
        else (row['tmsNumber_left'] if not pd.isna(row['tmsNumber_left'])
              else row['tmsNumber_right'])
    ),
    axis=1
)
gdf_combined['name'] = gdf_combined.apply(
    lambda row: (
        row['name_1'] if isinstance(row['name_1'], str)
        else (row['name_2'] if isinstance(row['name_2'], str)
              else row['name'])
    ),
    axis=1
)
gdf_combined = gdf_combined[['name','oneway','geometry','highway','tunnel','tmsNumber']]
gdf_combined = gdf_combined.reset_index(drop=True)
gdf_combined = gdf_combined.explode()

# -------------------------
# 5. Save Relevant Roads
# -------------------------
# Convert list-type columns to strings
for col in ['highway', 'name', 'oneway']:
    gdf_combined[col] = gdf_combined[col].apply(
        lambda x: ', '.join(x) if isinstance(x, list) else x
    )

gdf_combined.to_file("relevant_osm_roads2.geojson", driver="GeoJSON")

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
C:\Users\Dan\AppData\Local\Temp\ipykernel_2560\867104745.py:287: UserWarning: `keep_geom_type=True` in overlay resulted in 5 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  only_roads_with_stations_tunnels = gpd.overlay(
C:\Users\Dan\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py:3490: FutureWarning: The `op` parameter is deprecated and will be removed in a futur

In [7]:
import requests
import json
import pandas as pd
import geopandas as gpd
import osmnx as ox
import shapely
import numpy as np
from shapely.geometry import Point, LineString

# Functions from Santeri

def split_by_num_points(row, point_gdf):
    if not np.isnan(row.num_points):
        points = int(row.num_points)
        if points > 1 and row.oneway == True:
            splitter = shapely.geometry.MultiPoint([row.geometry.interpolate((i/4), normalized=True) for i in range(1, points)])
            split_line = shapely.ops.split(shapely.ops.snap(row.geometry, splitter, 0.1), splitter)
            for segment in split_line:
                if point_gdf[point_gdf['tmsNumber']==row.tmsNumber].intersects(segment).any():
                    return segment
        else:
            return row.geometry

def road_is_not_link(row):
    if type(row['highway']) == list:
        for road_type in row['highway']:
            if 'link' in road_type:
                return False
    elif 'link' in row['highway']:
        return False
    return True

def capture_roads_by_names(row, roadnames):
    flattened = []
    for r in roadnames:
        if str(r) == 'nan':
            continue
        if type(r) == list:
            for sub_r in r:
                flattened.append(sub_r)
        else:
            flattened.append(r)
    return any(roadname in str(row['name']) for roadname in flattened)

def merge_multilines(row):
    if row['geometry'].type == "MultiLineString":
        single_line = shapely.ops.linemerge(row['geometry'])
        return single_line
    return row['geometry']

# ----------------------------------
# 1. Load TMS Station Data
# ----------------------------------

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))

gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
gdf.crs = "EPSG:4326"

# Filter stations for Oulu area
oulu = pd.concat([
    gdf.loc[["oulu" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tobo testipiste" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["st815_lentokentäntie" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_rusko2" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_kiiminki" in c.lower() for c in list(gdf['name'])]]
])

oulu_stations = oulu[
    (oulu.name != "vt7_Treksilä") &
    (oulu.name != "Vt4_Oulu_Kontinkangas_Flex") &
    (oulu.name != "vt4_Oulu_Haukipudas") &
    (oulu.name != "vt4_Oulu_Kello") &
    (oulu.name != "Vt20_Oulu_Honkimaa_Flex")
]

tampere = oulu_stations.to_crs(4326)

# -------------------------------------
# 2. Download and Prepare Road Networks
# -------------------------------------

G = ox.graph_from_place(
    "Oulu, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G, strict=False)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
tunnels = gdf_edges[gdf_edges["tunnel"] == "yes"]

G = ox.graph_from_place(
    "Oulu, Finland",
    custom_filter='["highway"~"primary|secondary|motorway|trunk"]',
    simplify=False
)
G = ox.simplification.simplify_graph(G)
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.to_crs(3067)
gdf_edges = gdf_edges[gdf_edges.apply(road_is_not_link, axis=1)]

tunnels = tunnels.reset_index(drop=True)
simple_edges = gdf_edges.reset_index(drop=True)
simple_edges['tunnel'] = 'null'

# --------------------------------
# 3. Match Roads with TMS Stations
# --------------------------------

tampere = tampere.to_crs(3067)
tampere['buffer'] = tampere['geometry'].buffer(5)
tampere = tampere.set_geometry('buffer')

new_gdf = gpd.sjoin(
    simple_edges[['name','oneway','geometry','highway','tunnel']],
    tampere[['buffer','tmsNumber']],
    predicate='intersects'
).reset_index(drop=True)

only_roads_with_stations = simple_edges[
    simple_edges.apply(capture_roads_by_names,
                       args=([i for i in new_gdf['name']], ), axis=1)
    | simple_edges['name'].isnull()
]
only_roads_with_stations_tunnels = gpd.overlay(
    only_roads_with_stations[['name','oneway','geometry','highway']],
    tunnels[['tunnel','geometry']], how='union'
)
only_roads_with_stations_tunnels = only_roads_with_stations_tunnels.join(
    gpd.sjoin(tampere, only_roads_with_stations_tunnels)
      .groupby("index_right").size().rename("num_points"),
    how="left",
)

joined_station_in_tunnel = gpd.sjoin(
    only_roads_with_stations_tunnels,
    tampere[['buffer','tmsNumber']],
    op='intersects'
)
joined_station_in_tunnel['geometry'] = joined_station_in_tunnel.apply(merge_multilines, axis=1)

new_gdf = new_gdf.set_index('index_right', drop=True)
new_gdf.index.name = None

new_gdf = new_gdf.join(
    gpd.sjoin(tampere, new_gdf).groupby("index_right").size().rename("num_points"),
    how="left",
)
new_gdf.geometry = new_gdf.apply(split_by_num_points, axis=1, args=(tampere,))

test_buffer = new_gdf.copy()
test_buffer['geometry'] = new_gdf.buffer(40, single_sided=False)
only_roads_with_stations_tunnels.index.name = None
test = gpd.clip(
    test_buffer,
    only_roads_with_stations_tunnels[only_roads_with_stations_tunnels['tunnel'].isnull()]
)
both_lanes = gpd.overlay(test, tunnels, how='union')

just_one_tunnel = joined_station_in_tunnel[joined_station_in_tunnel['tunnel'] == 'yes']
new_gdf = new_gdf.overlay(just_one_tunnel, how='identity')
new_gdf['tunnel'] = new_gdf.apply(
    lambda row: "yes" if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' else np.nan, axis=1
)
new_gdf = new_gdf.rename(columns={
    'num_points_1': 'num_points',
    'name_1': 'name',
    'oneway_1': 'oneway',
    'highway_1': 'highway',
    'tmsNumber_1': 'tmsNumber'
})
new_gdf = new_gdf.drop(['name_2','oneway_2','tunnel_1','tunnel_2','highway_2','tmsNumber_2'], axis=1)

# --------------------------------------------
# 4. Build the Single-Lane Road Representation
# --------------------------------------------

new_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(50, single_sided=False), crs='epsg:3067')
other_buffer = gpd.GeoDataFrame(geometry=new_gdf.buffer(5, single_sided=False), crs='epsg:3067')
gdf_one_lane = both_lanes.clip(
    new_buffer.overlay(other_buffer, how='difference'), keep_geom_type=True
)

gdf_filter = gdf_one_lane.copy()
for i, row in gdf_one_lane.iterrows():
    if isinstance(row.geometry, shapely.geometry.collection.GeometryCollection):
        lines = [shape for shape in row.geometry if isinstance(shape, shapely.geometry.LineString)]
        merged = shapely.ops.linemerge(lines)
        gdf_filter.at[i, 'geometry'] = merged

tampere['buffer'] = tampere.buffer(165)
tampere = tampere.set_geometry('buffer')
gdf_filtered = gpd.sjoin(gdf_filter, tampere, op='intersects').set_geometry('geometry_left')
gdf_filtered.index.name = None
gdf_filtered['geometry'] = gdf_filtered['geometry_left']
gdf_filtered = gdf_filtered.set_geometry('geometry')

gdf_combined = new_gdf.append(gdf_filtered)
gdf_combined = gdf_combined[gdf_combined.geom_type != 'Point']

gdf_combined['tunnel'] = gdf_combined.apply(
    lambda row: "yes"
    if row['tunnel_2'] == 'yes' or row['tunnel_1'] == 'yes' or row['tunnel'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['oneway'] = gdf_combined.apply(
    lambda row: "yes"
    if row['oneway_2'] == 'yes' or row['oneway_1'] == 'yes' or row['oneway'] == 'yes'
    else np.nan, axis=1
)
gdf_combined['highway'] = gdf_combined.apply(
    lambda row: (
        row['highway_1'] if isinstance(row['highway_1'], (str, list))
        else (row['highway_2'] if isinstance(row['highway_2'], (str, list))
              else row['highway'])
    ),
    axis=1
)
gdf_combined['tmsNumber'] = gdf_combined.apply(
    lambda row: (
        row['tmsNumber']
        if not pd.isna(row['tmsNumber'])
        else (row['tmsNumber_left'] if not pd.isna(row['tmsNumber_left'])
              else row['tmsNumber_right'])
    ),
    axis=1
)
gdf_combined['name'] = gdf_combined.apply(
    lambda row: (
        row['name_1'] if isinstance(row['name_1'], str)
        else (row['name_2'] if isinstance(row['name_2'], str)
              else row['name'])
    ),
    axis=1
)
gdf_combined = gdf_combined[['name','oneway','geometry','highway','tunnel','tmsNumber']]
gdf_combined = gdf_combined.reset_index(drop=True)
gdf_combined = gdf_combined.explode()

# -------------------------
# 5. Save Relevant Roads
# -------------------------
gdf_combined.to_file("relevant_osm_roads.geojson", driver="GeoJSON")

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\osmnx\geocoder.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  gdf = gdf.append(_geocode_query_to_gdf(q, wr, by_osmid))
C:\Users\Dan\AppData\Local\Temp\ipykernel_2560\241970641.py:127: UserWarning: `keep_geom_type=True` in overlay resulted in 4 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  only_roads_with_stations_tunnels = gpd.overlay(
C:\Users\Dan\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py:3490: FutureWarning: The `op` parameter is deprecated and will be removed in a futur

## Railway

In [ ]:
import geopandas as gpd

# 1. Load and reproject everything to EPSG:3067
rail_gdf     = gpd.read_file('files_QGIS/rail.geojson').to_crs(epsg=3067)
tampere_gdf  = gpd.read_file('files_QGIS/TMS_Stations.geojson').to_crs(epsg=3067)
oulu_gdf     = gpd.read_file('files_QGIS/oulu_stations.geojson').to_crs(epsg=3067)

# 2. Build a 2000 m buffer around each station and union into a single polygon
tampere_buffer = tampere_gdf.geometry.buffer(2000).unary_union
oulu_buffer    = oulu_gdf.geometry.buffer(2000).unary_union

# 3. Filter rail lines by intersection with each buffer
rail_tampere = rail_gdf[rail_gdf.geometry.intersects(tampere_buffer)]
rail_oulu    = rail_gdf[rail_gdf.geometry.intersects(oulu_buffer)]

# 4. Save to separate GeoJSONs
rail_tampere.to_file('files_QGIS/rail_tampere.geojson', driver='GeoJSON')
rail_oulu   .to_file('files_QGIS/rail_oulu.geojson',   driver='GeoJSON')

## Buildings height

#### tampere

In [86]:
import requests
import json
import io
import geopandas as gpd
import pandas as pd
import shapely.geometry
from api_key import API_KEY  # Assuming API key is stored here

def add_z_to_buildings(row):
    """Add elevation data to building geometries"""
    xs, ys = row['geometry'].exterior.coords.xy
    points = [[x, y, row['pohjankorkeus']/1000] for x, y in zip(xs, ys)]
    return shapely.geometry.Polygon(points)

def get_building_data(api_key):
    """Fetch building data within 2000m of calculation points"""
    # Load calculation points and create buffer
    points_gdf = gpd.read_file('calculation_points.geojson').to_crs(epsg=3067)
    buffer_area = points_gdf.geometry.buffer(2000).unary_union
    bounds = buffer_area.bounds  # (minx, miny, maxx, maxy)

    # Construct API URL with bounding box
    url = (f"https://avoin-paikkatieto.maanmittauslaitos.fi/maastotiedot/features/v1/collections/rakennus/items"
           f"?api-key={api_key}&bbox={','.join(map(str, bounds))}"
           f"&bbox-crs=http://www.opengis.net/def/crs/EPSG/0/3067"
           f"&crs=http://www.opengis.net/def/crs/EPSG/0/3067")

    buildings_gdf = gpd.GeoDataFrame()
    
    while True:
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            buildings_json = json.load(io.StringIO(response.text))
            
            # Process current page
            page_gdf = gpd.GeoDataFrame.from_features(buildings_json["features"])
            buildings_gdf = pd.concat([buildings_gdf, page_gdf], ignore_index=True)
            
            # Check for next page
            next_link = next((link['href'] for link in buildings_json["links"] 
                            if link['rel'] == 'next'), None)
            if not next_link or next_link == url:
                break
            url = next_link

        except (requests.exceptions.Timeout, requests.exceptions.HTTPError) as e:
            print(f"Request failed: {e}")
            break

    if not buildings_gdf.empty:
        # Process geometries and elevations
        buildings_gdf.crs = "EPSG:3067"
        buildings_gdf['geometry'] = buildings_gdf.apply(add_z_to_buildings, axis=1)
        buildings_gdf['roof_elevation'] = (buildings_gdf['pohjankorkeus']/1000) + 10 * buildings_gdf['kerrosluku']
        
        # Save to file
        buildings_gdf.to_file("buildings_elevations.geojson", driver="GeoJSON")
    return buildings_gdf

# Execute the main function
if __name__ == "__main__":
    get_building_data(API_KEY)

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


#### Oulu

In [1]:
import requests
import json
import io
import geopandas as gpd
import pandas as pd
import shapely.geometry
from api_key import API_KEY  # Assuming API key is stored here

def add_z_to_buildings(row):
    """Add elevation data to building geometries"""
    xs, ys = row['geometry'].exterior.coords.xy
    points = [[x, y, row['pohjankorkeus']/1000] for x, y in zip(xs, ys)]
    return shapely.geometry.Polygon(points)

def get_building_data(api_key):
    """Fetch building data within 2000m of calculation points"""
    # Load calculation points and create buffer
    points_gdf = gpd.read_file('oulu_data2.geojson').to_crs(epsg=3067)
    buffer_area = points_gdf.geometry.buffer(2000).unary_union
    bounds = buffer_area.bounds  # (minx, miny, maxx, maxy)

    # Construct API URL with bounding box
    url = (f"https://avoin-paikkatieto.maanmittauslaitos.fi/maastotiedot/features/v1/collections/rakennus/items"
           f"?api-key={api_key}&bbox={','.join(map(str, bounds))}"
           f"&bbox-crs=http://www.opengis.net/def/crs/EPSG/0/3067"
           f"&crs=http://www.opengis.net/def/crs/EPSG/0/3067")

    buildings_gdf = gpd.GeoDataFrame()
    
    while True:
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            buildings_json = json.load(io.StringIO(response.text))
            
            # Process current page
            page_gdf = gpd.GeoDataFrame.from_features(buildings_json["features"])
            buildings_gdf = pd.concat([buildings_gdf, page_gdf], ignore_index=True)
            
            # Check for next page
            next_link = next((link['href'] for link in buildings_json["links"] 
                            if link['rel'] == 'next'), None)
            if not next_link or next_link == url:
                break
            url = next_link

        except (requests.exceptions.Timeout, requests.exceptions.HTTPError) as e:
            print(f"Request failed: {e}")
            break

    if not buildings_gdf.empty:
        # Process geometries and elevations
        buildings_gdf.crs = "EPSG:3067"
        buildings_gdf['geometry'] = buildings_gdf.apply(add_z_to_buildings, axis=1)
        buildings_gdf['roof_elevation'] = (buildings_gdf['pohjankorkeus']/1000) + 10 * buildings_gdf['kerrosluku']
        
        # Save to file
        buildings_gdf.to_file("buildings_elevations2.geojson", driver="GeoJSON")
    return buildings_gdf

# Execute the main function
if __name__ == "__main__":
    get_building_data(API_KEY)

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


## Create TMS stations

In [3]:
import requests
import json
import io
import geopandas as gpd
import pandas as pd

# Download TMS data
url = "https://tie.digitraffic.fi//api/tms/v1/stations"
response = requests.get(url)
stations_data = json.load(io.StringIO(response.text))

# Create GeoDataFrame from GeoJSON features
gdf = gpd.GeoDataFrame.from_features(stations_data["features"])
gdf.crs = "EPSG:4326"  # set the coordinate reference system

# Filter for Tampere
tampere = pd.concat([
    gdf.loc[["tampere" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tre" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["rautaharkko" in c.lower() for c in list(gdf['name'])]]
])
tampere_stations = tampere[tampere.name != "vt7_Treksilä"]

oulu = pd.concat([
    gdf.loc[["oulu" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tobo testipiste" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["st815_lentokentäntie" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_rusko2" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_kiiminki" in c.lower() for c in list(gdf['name'])]]
])

oulu_stations = oulu[
    (oulu.name != "vt7_Treksilä") &
    (oulu.name != "Vt4_Oulu_Kontinkangas_Flex") &
    (oulu.name != "vt4_Oulu_Haukipudas") &
    (oulu.name != "vt4_Oulu_Kello") &
    (oulu.name != "Vt20_Oulu_Honkimaa_Flex") &
    (oulu.name != "kt45_Oulunkylä")
]

oulu_stations = oulu_stations[
    (oulu.name != "vt4_Oulu_Kaakkuri_LML") &
    (oulu.name != "vt4_Oulu_Hiironen") &
    (oulu.name != "vt4_Oulu_Laanila") &
    (oulu.name != "vt4_Oulu_Mäntylä") &
    (oulu.name != "st815_Oulunsalo")
]


# Combine stations
tms_stations = pd.concat([tampere_stations, oulu_stations])

# tampere_stations.to_file("files_QGIS/Tampere_Stations_disp.geojson", driver='GeoJSON')
oulu_stations.to_file("files_QGIS/oulu_stations_disp.geojson", driver='GeoJSON')
# tms_stations.to_file("files_QGIS/TMS_Stations.shp")  # For Shapefile

c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\geopandas\geodataframe.py:1327: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)
c:\Users\Dan\anaconda3\envs\thesis\lib\site-packages\geopandas\io\file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


### Santeri's way

In [ ]:
stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))

gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
gdf.crs = "EPSG:4326"

# Filter stations for Tampere area
tampere = pd.concat([
    gdf.loc[["tampere" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tre" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["rautaharkko" in c.lower() for c in list(gdf['name'])]]
])

tampere.to_file("files_QGIS/TMS_Stations.geojson", driver='GeoJSON')

# Create traffic.csv from TMS stations

## year

### Tampere

In [ ]:
import requests
import json
import io
import geopandas as gpd
import datetime as dt
import pandas as pd

def read_csv_for_traffic(url):
    try:
        cols = [
            "tmsnumber","year","doy","h","m","s","ms",
            "pituus (m)","kaista","suunta","class",
            "nopeus (km/h)","faulty",
            "kokonaisaika (tekninen)",
            "aikaväli (tekninen)","jonoalku (tekninen)"
        ]
        return pd.read_csv(url, names=cols, sep=";")
    except Exception:
        return None

def create_traffic_df_by_hour(
    traffic_stations,
    first_day=dt.datetime(2022, 1, 1).timetuple().tm_yday,
    last_day=dt.datetime(2022, 12, 31).timetuple().tm_yday
) -> pd.DataFrame:
    tmsnumbers = traffic_stations['tmsNumber'].tolist()
    all_hours = []

    for doy in range(first_day, last_day + 1):
        urls = [
            f"https://tie.digitraffic.fi/api/tms/v1/history/raw/"
            f"lamraw_{tms}_{21 if tms == 464 else 22}_{doy}.csv"
            for tms in tmsnumbers
        ]

        # read & drop any fails
        dfs = [df for df in map(read_csv_for_traffic, urls) if df is not None and not df.empty]
        if not dfs:
            continue

        df = pd.concat(dfs, ignore_index=True)

        # bring year into full form
        df['year'] += 2000

        # build datetime: start from Jan 1 of that year, add day‐of‐year, hour, etc.
        base = pd.to_datetime(df['year'], format='%Y')
        df['date'] = (
            base
            + pd.to_timedelta(df['doy'] - 1, unit='D')
            + pd.to_timedelta(df['h'], unit='h')
            + pd.to_timedelta(df['m'], unit='m')
            + pd.to_timedelta(df['s'], unit='s')
            + pd.to_timedelta(df['ms'], unit='ms')
        )

        # drop faulty rows
        df = df[df['faulty'] == 0]

        # aggregate per hour, class, station
        hourly = (
            df
            .groupby([pd.Grouper(key='date', freq='H'), 'class', 'tmsnumber'])
            .agg(count=('nopeus (km/h)', 'size'),
                 speed=('nopeus (km/h)', 'mean'))
            .reset_index()
        )
        all_hours.append(hourly)

    if not all_hours:
        return pd.DataFrame()
    return pd.concat(all_hours, ignore_index=True)

# ——— Main ———

# 1. Station metadata
resp = requests.get("https://tie.digitraffic.fi/api/tms/v1/stations")
resp.raise_for_status()
stations = resp.json()

# 2. GeoDataFrame + filter exact numbers
gdf = gpd.GeoDataFrame.from_features(stations["features"])
gdf.crs = "EPSG:4326"
desired = [435, 438, 439, 449, 451, 452, 455, 456, 457, 458, 464, 471]
tampere = gdf[gdf['tmsNumber'].isin(desired)].copy()

# 3. Build hourly traffic DF
traffic_df_hour = create_traffic_df_by_hour(tampere)
traffic_df_hour = traffic_df_hour[traffic_df_hour['class'] != 0]

# Pivot to get counts and speeds per class
pivot = pd.pivot_table(traffic_df_hour, values=['count', 'speed'], index=['date', 'tmsnumber'], columns='class')
pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().fillna(0)

# Compute aggregated light and heavy
pivot['count_light'] = pivot[['count_1', 'count_6', 'count_7']].sum(axis=1)
pivot['count_heavy'] = pivot[['count_2', 'count_3', 'count_4', 'count_5', 'count_9']].sum(axis=1)
pivot['speed_light'] = pivot.apply(lambda row: 
    (row['count_1']*row['speed_1'] + row['count_6']*row['speed_6'] + row['count_7']*row['speed_7']) / row['count_light'] if row['count_light'] >0 else 0, axis=1)
pivot['speed_heavy'] = pivot.apply(lambda row: 
    (row['count_2']*row['speed_2'] + row['count_3']*row['speed_3'] + row['count_4']*row['speed_4'] + 
     row['count_5']*row['speed_5'] + row['count_9']*row['speed_9']) / row['count_heavy'] if row['count_heavy'] >0 else 0, axis=1)

# Classify day/night
pivot['period'] = pivot['date'].dt.hour.apply(lambda h: 'd' if 7 <= h < 22 else 'n')
pivot['date_day'] = pivot['date'].dt.date

# Aggregate by date_day, tmsnumber, period
classes = [1,2,3,4,5,6,7,9]
light_cols = ('count_1','count_6','count_7')
heavy_cols = ('count_2','count_3','count_4','count_5','count_9')

def aggregate_group(group):
    agg = {}
    for cls in classes:
        cnt_col = f"count_{cls}"
        spd_col = f"speed_{cls}"
        total_count = group[cnt_col].sum()
        agg[cnt_col] = total_count
        if total_count > 0:
            weighted_speed = (group[cnt_col] * group[spd_col]).sum() / total_count
        else:
            weighted_speed = 0
        agg[spd_col] = weighted_speed
    # Light
    total_light = group[list(light_cols)].sum().sum()
    agg['count_light'] = total_light
    if total_light > 0:
        weighted_speed_light = (group['count_1']*group['speed_1'] + group['count_6']*group['speed_6'] + group['count_7']*group['speed_7']).sum() / total_light
    else:
        weighted_speed_light = 0
    agg['speed_light'] = weighted_speed_light
    # Heavy
    total_heavy = group[list(heavy_cols)].sum().sum()
    agg['count_heavy'] = total_heavy
    if total_heavy > 0:
        weighted_speed_heavy = (group['count_2']*group['speed_2'] + group['count_3']*group['speed_3'] + group['count_4']*group['speed_4'] +
                               group['count_5']*group['speed_5'] + group['count_9']*group['speed_9']).sum() / total_heavy
    else:
        weighted_speed_heavy = 0
    agg['speed_heavy'] = weighted_speed_heavy
    return pd.Series(agg)

grouped = pivot.groupby(['date_day', 'tmsnumber', 'period']).apply(aggregate_group).reset_index()

# Pivot to have d and n columns
merged = grouped.pivot(index=['date_day', 'tmsnumber'], columns='period')
merged.columns = [f"{col[1]}_{col[0]}" for col in merged.columns]
merged = merged.reset_index()

# Format date and rename columns
merged['date'] = merged['date_day'].dt.strftime('%d.%m.%Y')
merged.rename(columns={'tmsnumber': 'tmsNumber'}, inplace=True)
merged.drop('date_day', axis=1, inplace=True)

# Ensure all columns are present
columns_order = ['date', 'tmsNumber',
    'd_count_1','d_count_2','d_count_3','d_count_4','d_count_5','d_count_6','d_count_7','d_count_9',
    'd_speed_1','d_speed_2','d_speed_3','d_speed_4','d_speed_5','d_speed_6','d_speed_7','d_speed_9',
    'd_count_light','d_count_heavy','d_speed_light','d_speed_heavy',
    'n_count_1','n_count_2','n_count_3','n_count_4','n_count_5','n_count_6','n_count_7','n_count_9',
    'n_speed_1','n_speed_2','n_speed_3','n_speed_4','n_speed_5','n_speed_6','n_speed_7','n_speed_9',
    'n_count_light','n_count_heavy','n_speed_light','n_speed_heavy'
]
for col in columns_order:
    if col not in merged.columns:
        merged[col] = 0
merged = merged[columns_order]

# Save
merged.to_csv('files_QGIS/traffic_tampere_year.csv', index=False)

In [7]:
print("Unique vehicle classes:", sorted(traffic_df_hour['class'].unique()))

Unique vehicle classes: [1, 2, 3, 4, 5, 6, 7, 9]


In [9]:
traffic_df_hour

,date,class,tmsnumber,count,speed
1,2022-01-01 00:00:00,1,435,506,96.869565
2,2022-01-01 00:00:00,1,438,499,62.779559
3,2022-01-01 00:00:00,1,439,221,57.895928
4,2022-01-01 00:00:00,1,449,463,89.159827
5,2022-01-01 00:00:00,1,451,176,84.232955
...,...,...,...,...,...
100072,2022-02-26 23:00:00,6,435,5,94.600000
100073,2022-02-26 23:00:00,6,438,1,72.000000
100074,2022-02-26 23:00:00,6,439,1,45.000000
100075,2022-02-26 23:00:00,6,449,3,78.333333


In [10]:
import joblib

# Save the model
joblib.dump(traffic_df_hour, 'traffic_df_hour.pkl')

['traffic_df_hour.pkl']

In [11]:
# Pivot to get counts and speeds per class
pivot = pd.pivot_table(traffic_df_hour, values=['count', 'speed'], index=['date', 'tmsnumber'], columns='class')
pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().fillna(0)

# Compute aggregated light and heavy
pivot['count_light'] = pivot[['count_1', 'count_6', 'count_7']].sum(axis=1)
pivot['count_heavy'] = pivot[['count_2', 'count_3', 'count_4', 'count_5', 'count_9']].sum(axis=1)
pivot['speed_light'] = pivot.apply(lambda row: 
    (row['count_1']*row['speed_1'] + row['count_6']*row['speed_6'] + row['count_7']*row['speed_7']) / row['count_light'] if row['count_light'] >0 else 0, axis=1)
pivot['speed_heavy'] = pivot.apply(lambda row: 
    (row['count_2']*row['speed_2'] + row['count_3']*row['speed_3'] + row['count_4']*row['speed_4'] + 
     row['count_5']*row['speed_5'] + row['count_9']*row['speed_9']) / row['count_heavy'] if row['count_heavy'] >0 else 0, axis=1)

# Classify day/night
pivot['period'] = pivot['date'].dt.hour.apply(lambda h: 'd' if 7 <= h < 22 else 'n')
pivot['date_day'] = pivot['date'].dt.date

# Aggregate by date_day, tmsnumber, period
classes = [1,2,3,4,5,6,7,9]
light_cols = ('count_1','count_6','count_7')
heavy_cols = ('count_2','count_3','count_4','count_5','count_9')

def aggregate_group(group):
    agg = {}
    for cls in classes:
        cnt_col = f"count_{cls}"
        spd_col = f"speed_{cls}"
        total_count = group[cnt_col].sum()
        agg[cnt_col] = total_count
        if total_count > 0:
            weighted_speed = (group[cnt_col] * group[spd_col]).sum() / total_count
        else:
            weighted_speed = 0
        agg[spd_col] = weighted_speed
    # Light
    total_light = group[list(light_cols)].sum().sum()
    agg['count_light'] = total_light
    if total_light > 0:
        weighted_speed_light = (group['count_1']*group['speed_1'] + group['count_6']*group['speed_6'] + group['count_7']*group['speed_7']).sum() / total_light
    else:
        weighted_speed_light = 0
    agg['speed_light'] = weighted_speed_light
    # Heavy
    total_heavy = group[list(heavy_cols)].sum().sum()
    agg['count_heavy'] = total_heavy
    if total_heavy > 0:
        weighted_speed_heavy = (group['count_2']*group['speed_2'] + group['count_3']*group['speed_3'] + group['count_4']*group['speed_4'] +
                               group['count_5']*group['speed_5'] + group['count_9']*group['speed_9']).sum() / total_heavy
    else:
        weighted_speed_heavy = 0
    agg['speed_heavy'] = weighted_speed_heavy
    return pd.Series(agg)

grouped = pivot.groupby(['date_day', 'tmsnumber', 'period']).apply(aggregate_group).reset_index()

# Pivot to have d and n columns
merged = grouped.pivot(index=['date_day', 'tmsnumber'], columns='period')
merged.columns = [f"{col[1]}_{col[0]}" for col in merged.columns]
merged = merged.reset_index()

# Format date and rename columns
merged['date'] = pd.to_datetime(merged['date_day']).dt.strftime('%d.%m.%Y')
merged.rename(columns={'tmsnumber': 'tmsNumber'}, inplace=True)
merged.drop('date_day', axis=1, inplace=True)

# Ensure all columns are present
columns_order = ['date', 'tmsNumber',
    'd_count_1','d_count_2','d_count_3','d_count_4','d_count_5','d_count_6','d_count_7','d_count_9',
    'd_speed_1','d_speed_2','d_speed_3','d_speed_4','d_speed_5','d_speed_6','d_speed_7','d_speed_9',
    'd_count_light','d_count_heavy','d_speed_light','d_speed_heavy',
    'n_count_1','n_count_2','n_count_3','n_count_4','n_count_5','n_count_6','n_count_7','n_count_9',
    'n_speed_1','n_speed_2','n_speed_3','n_speed_4','n_speed_5','n_speed_6','n_speed_7','n_speed_9',
    'n_count_light','n_count_heavy','n_speed_light','n_speed_heavy'
]
for col in columns_order:
    if col not in merged.columns:
        merged[col] = 0
merged = merged[columns_order]

# Save
merged.to_csv('files_QGIS/traffic_tampere_year.csv', index=False)

In [16]:
merged

,date,tmsNumber,d_count_1,d_count_2,d_count_3,d_count_4,d_count_5,d_count_6,d_count_7,d_count_9,...,n_speed_3,n_speed_4,n_speed_5,n_speed_6,n_speed_7,n_speed_9,n_count_light,n_count_heavy,n_speed_light,n_speed_heavy
0,02.01.2021,464,6149.0,24.0,41.0,3.0,30.0,52.0,2.0,0.0,...,77.250000,68.500000,78.333333,72.428571,73.000000,0.00000,354.0,15.0,76.935028,75.466667
1,03.01.2021,464,4827.0,26.0,26.0,2.0,16.0,39.0,1.0,0.0,...,80.333333,67.000000,65.000000,72.000000,52.500000,0.00000,343.0,14.0,72.303207,65.857143
2,04.01.2021,464,9019.0,200.0,46.0,18.0,58.0,97.0,17.0,0.0,...,76.333333,74.666667,76.400000,69.500000,61.666667,0.00000,876.0,60.0,74.954338,74.916667
3,05.01.2021,464,9165.0,187.0,52.0,25.0,56.0,86.0,16.0,0.0,...,75.166667,79.500000,72.750000,77.222222,72.111111,0.00000,925.0,68.0,76.870270,74.426471
4,06.01.2021,464,5276.0,47.0,33.0,4.0,38.0,41.0,4.0,0.0,...,0.000000,78.000000,76.000000,71.500000,76.000000,0.00000,313.0,7.0,78.597444,76.857143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670,25.02.2022,471,38198.0,965.0,140.0,715.0,722.0,452.0,112.0,168.0,...,81.631579,80.600000,80.609091,87.433333,83.538462,79.54321,3910.0,574.0,87.760614,80.649826
671,26.02.2022,435,30963.0,286.0,47.0,141.0,152.0,513.0,87.0,0.0,...,88.785714,82.057143,81.516393,91.475410,85.500000,0.00000,3032.0,246.0,97.877968,83.186992
672,26.02.2022,438,28980.0,127.0,395.0,21.0,95.0,200.0,12.0,0.0,...,63.564706,70.000000,64.826087,68.125000,68.500000,0.00000,2982.0,148.0,66.569081,64.466216
673,26.02.2022,439,14874.0,146.0,388.0,32.0,26.0,115.0,13.0,0.0,...,44.061224,51.125000,57.500000,51.857143,45.000000,0.00000,1438.0,87.0,57.867177,48.218391


In [14]:
# Group by tmsNumber and average everything else
station_avg = merged.groupby('tmsNumber').mean(numeric_only=True).reset_index()
station_avg

,tmsNumber,d_count_1,d_count_2,d_count_3,d_count_4,d_count_5,d_count_6,d_count_7,d_count_9,d_speed_1,...,n_speed_3,n_speed_4,n_speed_5,n_speed_6,n_speed_7,n_speed_9,n_count_light,n_count_heavy,n_speed_light,n_speed_heavy
0,435,34091.263158,1164.403509,42.701754,340.526316,626.087719,348.666667,71.017544,0.000000,94.695142,...,85.064421,82.997870,82.011537,88.391717,84.294614,0.000000,3933.105263,423.912281,96.969662,82.874294
1,438,32276.473684,411.122807,450.771930,58.210526,129.052632,181.368421,32.666667,0.000000,63.063969,...,63.038180,64.988787,65.141154,63.001467,57.839482,0.000000,3330.000000,209.701754,66.464583,63.749357
2,439,16157.421053,327.052632,351.280702,70.228070,47.736842,87.508772,20.070175,0.000000,53.488235,...,45.274328,48.116096,57.708278,52.432143,32.556140,0.000000,1418.210526,108.526316,54.901843,47.733790
3,449,38926.596491,1468.421053,65.649123,581.192982,900.157895,402.508772,125.017544,0.000000,86.836643,...,79.690306,81.731423,81.045828,84.117369,83.140520,0.000000,4459.368421,673.298246,91.796608,81.634651
4,451,13362.964286,441.696429,43.000000,145.357143,451.642857,312.196429,57.142857,0.000000,81.402952,...,80.239092,81.888490,81.941591,83.897392,83.981620,0.000000,1384.250000,213.785714,84.806826,81.867067
5,452,26046.535714,311.035714,22.339286,75.500000,77.982143,137.125000,30.803571,0.000000,60.196178,...,58.667955,58.574841,61.301015,60.862559,44.701327,0.000000,2895.142857,116.767857,62.037778,59.946006
6,455,17376.839286,266.839286,123.678571,42.232143,96.000000,153.750000,25.071429,0.000000,71.223299,...,65.791356,68.571104,70.012844,69.431290,57.787835,0.000000,1761.464286,101.500000,72.779553,67.808744
7,456,32612.875000,399.964286,480.875000,150.642857,95.339286,172.107143,31.071429,0.000000,56.064677,...,53.327811,55.437396,58.405355,56.966506,46.274212,0.000000,3025.464286,216.321429,59.135842,55.155245
8,457,25090.428571,415.892857,140.785714,57.625000,127.214286,195.071429,43.803571,0.000000,61.824290,...,61.645538,60.690517,64.144363,62.720642,53.175749,0.000000,2587.660714,119.732143,63.897581,61.339285
9,458,13874.767857,382.142857,224.625000,278.875000,60.000000,86.892857,112.410714,12.732143,65.956774,...,63.958167,64.295633,67.581322,68.738141,65.335713,77.942434,1191.232143,127.535714,68.543377,64.552238


In [15]:
station_avg.to_csv('files_QGIS/traffic_tampere_year.csv', index=False)

### Oulu

In [1]:
import requests
import json
import io
import geopandas as gpd
import datetime as dt
import pandas as pd

def read_csv_for_traffic(url):
    try:
        cols = [
            "tmsnumber","year","doy","h","m","s","ms",
            "pituus (m)","kaista","suunta","class",
            "nopeus (km/h)","faulty",
            "kokonaisaika (tekninen)",
            "aikaväli (tekninen)","jonoalku (tekninen)"
        ]
        return pd.read_csv(url, names=cols, sep=";")
    except Exception:
        return None

def create_traffic_df_by_hour(
    traffic_stations,
    first_day=dt.datetime(2022, 1, 1).timetuple().tm_yday,
    last_day=dt.datetime(2022, 12, 31).timetuple().tm_yday
) -> pd.DataFrame:
    tmsnumbers = traffic_stations['tmsNumber'].tolist()
    all_hours = []

    for doy in range(first_day, last_day + 1):
        urls = [
            f"https://tie.digitraffic.fi/api/tms/v1/history/raw/"
            f"lamraw_{tms}_{21 if tms == 1223 else 22}_{doy}.csv"
            for tms in tmsnumbers
        ]

        # read & drop any fails
        dfs = [df for df in map(read_csv_for_traffic, urls) if df is not None and not df.empty]
        if not dfs:
            continue

        df = pd.concat(dfs, ignore_index=True)

        # bring year into full form
        df['year'] += 2000

        # build datetime: start from Jan 1 of that year, add day‐of‐year, hour, etc.
        base = pd.to_datetime(df['year'], format='%Y')
        df['date'] = (
            base
            + pd.to_timedelta(df['doy'] - 1, unit='D')
            + pd.to_timedelta(df['h'], unit='h')
            + pd.to_timedelta(df['m'], unit='m')
            + pd.to_timedelta(df['s'], unit='s')
            + pd.to_timedelta(df['ms'], unit='ms')
        )

        # drop faulty rows
        df = df[df['faulty'] == 0]

        # aggregate per hour, class, station
        hourly = (
            df
            .groupby([pd.Grouper(key='date', freq='H'), 'class', 'tmsnumber'])
            .agg(count=('nopeus (km/h)', 'size'),
                 speed=('nopeus (km/h)', 'mean'))
            .reset_index()
        )
        all_hours.append(hourly)

    if not all_hours:
        return pd.DataFrame()
    return pd.concat(all_hours, ignore_index=True)

# ——— Main ———

# 1. Station metadata
resp = requests.get("https://tie.digitraffic.fi/api/tms/v1/stations")
resp.raise_for_status()
stations = resp.json()

# 2. GeoDataFrame + filter exact numbers
gdf = gpd.GeoDataFrame.from_features(stations["features"])
gdf.crs = "EPSG:4326"
desired = [1257, 1256, 1239, 1250, 1251, 1206, 1223, 1238, 1246, 1237, 1247, 1244, 1254, 21201]
tampere = gdf[gdf['tmsNumber'].isin(desired)].copy()

# 3. Build hourly traffic DF
traffic_df_hour = create_traffic_df_by_hour(tampere)
traffic_df_hour = traffic_df_hour[traffic_df_hour['class'] != 0]

In [2]:
# Pivot to get counts and speeds per class
pivot = pd.pivot_table(traffic_df_hour, values=['count', 'speed'], index=['date', 'tmsnumber'], columns='class')
pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().fillna(0)

# Compute aggregated light and heavy
pivot['count_light'] = pivot[['count_1', 'count_6', 'count_7']].sum(axis=1)
pivot['count_heavy'] = pivot[['count_2', 'count_3', 'count_4', 'count_5', 'count_9']].sum(axis=1)
pivot['speed_light'] = pivot.apply(lambda row: 
    (row['count_1']*row['speed_1'] + row['count_6']*row['speed_6'] + row['count_7']*row['speed_7']) / row['count_light'] if row['count_light'] >0 else 0, axis=1)
pivot['speed_heavy'] = pivot.apply(lambda row: 
    (row['count_2']*row['speed_2'] + row['count_3']*row['speed_3'] + row['count_4']*row['speed_4'] + 
     row['count_5']*row['speed_5'] + row['count_9']*row['speed_9']) / row['count_heavy'] if row['count_heavy'] >0 else 0, axis=1)

# Classify day/night
pivot['period'] = pivot['date'].dt.hour.apply(lambda h: 'd' if 7 <= h < 22 else 'n')
pivot['date_day'] = pivot['date'].dt.date

# Aggregate by date_day, tmsnumber, period
classes = [1,2,3,4,5,6,7,9]
light_cols = ('count_1','count_6','count_7')
heavy_cols = ('count_2','count_3','count_4','count_5','count_9')

def aggregate_group(group):
    agg = {}
    for cls in classes:
        cnt_col = f"count_{cls}"
        spd_col = f"speed_{cls}"
        total_count = group[cnt_col].sum()
        agg[cnt_col] = total_count
        if total_count > 0:
            weighted_speed = (group[cnt_col] * group[spd_col]).sum() / total_count
        else:
            weighted_speed = 0
        agg[spd_col] = weighted_speed
    # Light
    total_light = group[list(light_cols)].sum().sum()
    agg['count_light'] = total_light
    if total_light > 0:
        weighted_speed_light = (group['count_1']*group['speed_1'] + group['count_6']*group['speed_6'] + group['count_7']*group['speed_7']).sum() / total_light
    else:
        weighted_speed_light = 0
    agg['speed_light'] = weighted_speed_light
    # Heavy
    total_heavy = group[list(heavy_cols)].sum().sum()
    agg['count_heavy'] = total_heavy
    if total_heavy > 0:
        weighted_speed_heavy = (group['count_2']*group['speed_2'] + group['count_3']*group['speed_3'] + group['count_4']*group['speed_4'] +
                               group['count_5']*group['speed_5'] + group['count_9']*group['speed_9']).sum() / total_heavy
    else:
        weighted_speed_heavy = 0
    agg['speed_heavy'] = weighted_speed_heavy
    return pd.Series(agg)

grouped = pivot.groupby(['date_day', 'tmsnumber', 'period']).apply(aggregate_group).reset_index()

# Pivot to have d and n columns
merged = grouped.pivot(index=['date_day', 'tmsnumber'], columns='period')
merged.columns = [f"{col[1]}_{col[0]}" for col in merged.columns]
merged = merged.reset_index()

# Format date and rename columns
merged['date'] = pd.to_datetime(merged['date_day']).dt.strftime('%d.%m.%Y')
merged.rename(columns={'tmsnumber': 'tmsNumber'}, inplace=True)
merged.drop('date_day', axis=1, inplace=True)

# Ensure all columns are present
columns_order = ['date', 'tmsNumber',
    'd_count_1','d_count_2','d_count_3','d_count_4','d_count_5','d_count_6','d_count_7','d_count_9',
    'd_speed_1','d_speed_2','d_speed_3','d_speed_4','d_speed_5','d_speed_6','d_speed_7','d_speed_9',
    'd_count_light','d_count_heavy','d_speed_light','d_speed_heavy',
    'n_count_1','n_count_2','n_count_3','n_count_4','n_count_5','n_count_6','n_count_7','n_count_9',
    'n_speed_1','n_speed_2','n_speed_3','n_speed_4','n_speed_5','n_speed_6','n_speed_7','n_speed_9',
    'n_count_light','n_count_heavy','n_speed_light','n_speed_heavy'
]
for col in columns_order:
    if col not in merged.columns:
        merged[col] = 0
merged = merged[columns_order]

# Save
merged.to_csv('files_QGIS/traffic_oulu_year.csv', index=False)

In [3]:
# Group by tmsNumber and average everything else
station_avg = merged.groupby('tmsNumber').mean(numeric_only=True).reset_index()
station_avg.to_csv('files_QGIS/traffic_oulu_year.csv', index=False)

In [ ]:
station_avg

,tmsNumber,d_count_1,d_count_2,d_count_3,d_count_4,d_count_5,d_count_6,d_count_7,d_count_9,d_speed_1,...,n_speed_3,n_speed_4,n_speed_5,n_speed_6,n_speed_7,n_speed_9,n_count_light,n_count_heavy,n_speed_light,n_speed_heavy
0,1206,20690.551724,317.724138,34.620690,127.965517,270.931034,366.620690,64.137931,39.068966,99.166240,...,81.695325,84.775754,83.105555,94.899229,88.161681,83.613813,2689.517241,262.034483,100.867573,84.014325
1,1223,9951.034483,197.172414,69.724138,26.965517,171.034483,260.655172,37.448276,0.000000,63.839290,...,63.695250,63.292066,65.703008,65.284210,63.353065,0.000000,1204.965517,121.241379,65.507128,65.036125
2,1237,38400.137931,837.965517,88.310345,285.344828,679.000000,561.206897,94.034483,70.103448,93.006695,...,80.145273,82.142822,78.633155,90.532690,82.874885,80.224084,4629.758621,516.620690,95.200253,80.217134
3,1238,27706.758621,339.655172,44.275862,192.379310,383.448276,346.827586,50.551724,48.551724,98.224273,...,79.904046,84.145077,82.490641,94.076382,86.151051,82.430443,3095.137931,267.344828,99.777504,83.308817
4,1239,10962.035714,134.035714,127.964286,9.464286,19.178571,137.250000,13.607143,0.000000,74.717047,...,74.850810,67.762500,76.323643,76.857439,37.274235,0.000000,1303.000000,43.392857,78.340937,74.873984
5,1244,14177.071429,197.000000,19.500000,103.392857,243.107143,181.678571,28.750000,32.964286,95.388935,...,76.664286,82.757613,81.069425,92.139885,84.113053,79.972171,1909.714286,116.392857,96.473753,82.014626
6,1246,21592.821429,291.071429,47.464286,188.750000,382.500000,307.928571,52.500000,54.750000,98.853476,...,80.836179,84.244443,83.124701,95.105997,85.488845,82.747937,2479.892857,249.714286,100.336715,83.458760
7,1247,9281.785714,156.357143,120.714286,10.642857,38.285714,116.535714,18.214286,0.000000,57.101852,...,57.992323,56.782738,59.236650,55.192014,51.995174,0.000000,626.071429,50.178571,61.922758,59.630055
8,1250,25730.357143,716.392857,47.428571,261.571429,642.714286,465.678571,114.107143,104.714286,98.531132,...,82.762412,84.794735,83.318070,94.680554,85.468208,84.260712,3167.107143,474.857143,99.959508,84.392804
9,1251,30319.857143,855.214286,198.964286,279.071429,714.464286,493.571429,128.214286,0.000000,96.420024,...,87.382367,84.780481,83.855996,89.928021,89.501615,0.000000,3758.714286,514.071429,97.873078,85.137554


## For one day

#### Tampere

In [ ]:
import requests
import json
import io
import geopandas as gpd
import datetime as dt
import numpy as np
import pandas as pd


def compose_date(years, months=1, days=1, weeks=None, hours=None, minutes=None,
                 seconds=None, milliseconds=None, microseconds=None, nanoseconds=None):
    years = np.asarray(years) - 1970
    months = np.asarray(months) - 1
    days = np.asarray(days) - 1
    types = ('<M8[Y]', '<m8[M]', '<m8[D]', '<m8[W]', '<m8[h]',
             '<m8[m]', '<m8[s]', '<m8[ms]', '<m8[us]', '<m8[ns]')
    vals = (years, months, days, weeks, hours, minutes, seconds,
            milliseconds, microseconds, nanoseconds)
    return sum(np.asarray(v, dtype=t) for t, v in zip(types, vals)
               if v is not None)


def read_csv_for_traffic(url):
    try:
        #dtypes = {"tmsNumber": int, "year": int, "doy": int, "h": int, "m": int, "s": int, "ms": int, "pituus (m)": float, "kaista": int, "suunta": int, "class": int, "nopeus (km/h)": float, "faulty": int, "kokonaisaika (tekninen)": int, "aikaväli (tekninen)": int, "jonoalku (tekninen)": int}
        traffic_cols = ["tmsnumber","year","doy","h","m","s","ms","pituus (m)","kaista","suunta","class","nopeus (km/h)","faulty","kokonaisaika (tekninen)","aikaväli (tekninen)","jonoalku (tekninen)"]
        return pd.read_csv(url, names=traffic_cols, sep=";")
    except:
        return None


def create_traffic_df_by_hour(traffic_stations, first_day=dt.datetime(2022, 9, 9).timetuple().tm_yday, last_day=dt.datetime(2022, 9, 10).timetuple().tm_yday):
    tmsnumbers = traffic_stations['tmsNumber'].values.tolist()

    traffic_by_hour = []
    
    for i in range(first_day, last_day):
        urls = (f"https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_464_21_253.csv" if tmsnumber == 464 else f"https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_{tmsnumber}_22_{i}.csv" for tmsnumber in tmsnumbers)
        
        traffic_df = pd.concat(map(read_csv_for_traffic, urls))
        if traffic_df is None:
            continue
        traffic_df['year'] = traffic_df['year'] + 2000
        traffic_df['date'] = compose_date(traffic_df['year'], days=traffic_df['doy'], hours=traffic_df['h'], minutes=traffic_df['m'], seconds=traffic_df['s'], milliseconds=traffic_df['ms'])
        traffic_df = traffic_df[traffic_df['faulty'] == 0]
        
        # Group by  hour
        result_df = traffic_df.groupby([pd.Grouper(key='date', freq='H'), traffic_df['class'], traffic_df['tmsnumber']]).mean()['nopeus (km/h)']
        result_df = result_df.reset_index()

        # Count the number of records per hour
        by_hour = traffic_df.groupby([pd.Grouper(key='date', freq='H'), traffic_df['class'], traffic_df['tmsnumber']]).size().reset_index(name='count')
        by_hour['speed'] = result_df['nopeus (km/h)']
        
        traffic_by_hour.append(by_hour)
    
    return pd.concat(traffic_by_hour, ignore_index=True)

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))
weather = requests.get("https://tie.digitraffic.fi/api/weather/v1/stations")
weather_t = json.load(io.StringIO(weather.text))
gdf_weather = gpd.GeoDataFrame.from_features(weather_t["features"])
gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
#ax = gdf_weather.plot()
gdf.crs = "EPSG:4326"
tampere = pd.concat([gdf.loc[["tampere" in c.lower() for c in  list(gdf['name'])]], gdf.loc[["tre" in c.lower() for c in  list(gdf['name'])]], gdf.loc[["rautaharkko" in c.lower() for c in  list(gdf['name'])]]])
# Not in Tampere
tampere = tampere[tampere.name != "vt7_Treksilä"] 
tampere = tampere[tampere.name != "vt3_Tampere_Myllypuro"]

tampere = tampere.to_crs(4326)

# Create hourly traffic dataframe
traffic_df_hour = create_traffic_df_by_hour(tampere)
traffic_df_hour = traffic_df_hour[traffic_df_hour['class'] != 0]

# Pivot to get counts and speeds per class in separate columns
pivot = pd.pivot_table(traffic_df_hour, values=['count', 'speed'], index=['date', 'tmsnumber'], columns='class')
# Flatten MultiIndex columns
pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().fillna(0)

# Compute aggregated counts and weighted speeds for light and heavy vehicles
# Here we consider classes 1,6,7 as light; and classes 2,3,4,5,9 as heavy.
pivot['count_light'] = pivot[['count_1', 'count_6', 'count_7']].sum(axis=1)
pivot['count_heavy'] = pivot[['count_2', 'count_3', 'count_4', 'count_5', 'count_9']].sum(axis=1)
pivot['speed_light'] = pivot.apply(lambda row: 
                                   (row['count_1'] * row['speed_1'] + row['count_6'] * row['speed_6'] + row['count_7'] * row['speed_7']) / row['count_light']
                                   if row['count_light'] > 0 else 0, axis=1)
pivot['speed_heavy'] = pivot.apply(lambda row: 
                                   (row['count_2'] * row['speed_2'] + row['count_3'] * row['speed_3'] + row['count_4'] * row['speed_4'] + 
                                    row['count_5'] * row['speed_5'] + row['count_9'] * row['speed_9']) / row['count_heavy']
                                   if row['count_heavy'] > 0 else 0, axis=1)

# Assign each row to a day or night period.
# Day: hour between 07:00 (inclusive) and 22:00 (exclusive), otherwise Night.
pivot['period'] = pivot['date'].dt.hour.apply(lambda h: 'd' if 7 <= h < 22 else 'n')

# List of classes to aggregate
classes = [1,2,3,4,5,6,7,9]
# List for light and heavy aggregates.
light_cols = ('count_1','count_6','count_7')
heavy_cols = ('count_2','count_3','count_4','count_5','count_9')

# Function to aggregate a group (per tmsnumber and period)
def aggregate_group(group):
    agg = {}
    # For each vehicle class, sum counts and compute weighted average speed.
    for cls in classes:
        cnt_col = f"count_{cls}"
        spd_col = f"speed_{cls}"
        total_count = group[cnt_col].sum()
        agg[cnt_col] = total_count
        # Weighted average speed; if total_count is zero, then 0.
        if total_count > 0:
            weighted_speed = (group[cnt_col] * group[spd_col]).sum() / total_count
        else:
            weighted_speed = 0
        agg[spd_col] = weighted_speed
    # Aggregated light vehicles
    total_light = group[list(light_cols)].sum().sum()
    agg['count_light'] = total_light
    if total_light > 0:
        weighted_speed_light = (group['count_1'] * group['speed_1'] + 
                                group['count_6'] * group['speed_6'] + 
                                group['count_7'] * group['speed_7']).sum() / total_light
    else:
        weighted_speed_light = 0
    agg['speed_light'] = weighted_speed_light

    # Aggregated heavy vehicles
    total_heavy = group[list(heavy_cols)].sum().sum()
    agg['count_heavy'] = total_heavy
    if total_heavy > 0:
        weighted_speed_heavy = (group['count_2'] * group['speed_2'] + 
                                group['count_3'] * group['speed_3'] +
                                group['count_4'] * group['speed_4'] +
                                group['count_5'] * group['speed_5'] +
                                group['count_9'] * group['speed_9']).sum() / total_heavy
    else:
        weighted_speed_heavy = 0
    agg['speed_heavy'] = weighted_speed_heavy

    return pd.Series(agg)


# Group by tmsnumber and period to merge hourly data into day and night aggregates.
grouped = pivot.groupby(['tmsnumber', 'period']).apply(aggregate_group).reset_index()

# Pivot the groups so that day (d) and night (n) aggregates become separate columns.
merged = grouped.pivot(index='tmsnumber', columns='period')
# Flatten the columns: e.g., ("count_1", "d") becomes "d_count_1"
merged.columns = [f"{col[1]}_{col[0]}" for col in merged.columns]
merged = merged.reset_index()

# Rename column to "tmsNumber"
merged.rename(columns={'tmsnumber': 'tmsNumber'}, inplace=True)

# Add date column: For tmsNumber==464, date is "10.09.2021", else "09.09.2022".
merged['date'] = merged['tmsNumber'].apply(lambda x: "10.09.2021" if x == 464 else "09.09.2022")

# Reorder columns as specified in the output.
# Desired order:
columns_order = ['date', 'tmsNumber',
    'd_count_1','d_count_2','d_count_3','d_count_4','d_count_5','d_count_6','d_count_7','d_count_9',
    'd_speed_1','d_speed_2','d_speed_3','d_speed_4','d_speed_5','d_speed_6','d_speed_7','d_speed_9',
    'd_count_light','d_count_heavy','d_speed_light','d_speed_heavy',
    'n_count_1','n_count_2','n_count_3','n_count_4','n_count_5','n_count_6','n_count_7','n_count_9',
    'n_speed_1','n_speed_2','n_speed_3','n_speed_4','n_speed_5','n_speed_6','n_speed_7','n_speed_9',
    'n_count_light','n_count_heavy','n_speed_light','n_speed_heavy'
]
# Some columns might be missing if there were no counts in a given period – fill missing columns with 0.
for col in columns_order:
    if col not in merged.columns:
        merged[col] = 0

merged = merged[columns_order]

# Save the result to CSV.
merged.to_csv('files_QGIS/traffic.csv', index=False)

KeyError: "['count_8'] not in index"

#### Oulu

In [ ]:
import requests
import json
import io
import geopandas as gpd
import datetime as dt
import numpy as np
import pandas as pd


def compose_date(years, months=1, days=1, weeks=None, hours=None, minutes=None,
                 seconds=None, milliseconds=None, microseconds=None, nanoseconds=None):
    years = np.asarray(years) - 1970
    months = np.asarray(months) - 1
    days = np.asarray(days) - 1
    types = ('<M8[Y]', '<m8[M]', '<m8[D]', '<m8[W]', '<m8[h]',
             '<m8[m]', '<m8[s]', '<m8[ms]', '<m8[us]', '<m8[ns]')
    vals = (years, months, days, weeks, hours, minutes, seconds,
            milliseconds, microseconds, nanoseconds)
    return sum(np.asarray(v, dtype=t) for t, v in zip(types, vals)
               if v is not None)


def read_csv_for_traffic(url):
    try:
        #dtypes = {"tmsNumber": int, "year": int, "doy": int, "h": int, "m": int, "s": int, "ms": int, "pituus (m)": float, "kaista": int, "suunta": int, "class": int, "nopeus (km/h)": float, "faulty": int, "kokonaisaika (tekninen)": int, "aikaväli (tekninen)": int, "jonoalku (tekninen)": int}
        traffic_cols = ["tmsnumber","year","doy","h","m","s","ms","pituus (m)","kaista","suunta","class","nopeus (km/h)","faulty","kokonaisaika (tekninen)","aikaväli (tekninen)","jonoalku (tekninen)"]
        return pd.read_csv(url, names=traffic_cols, sep=";")
    except:
        return None


def create_traffic_df_by_hour(traffic_stations, first_day=dt.datetime(2022, 9, 9).timetuple().tm_yday, last_day=dt.datetime(2022, 9, 10).timetuple().tm_yday):
    tmsnumbers = traffic_stations['tmsNumber'].values.tolist()

    traffic_by_hour = []
    
    for i in range(first_day, last_day):
        urls = (f"https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_1223_21_253.csv" if tmsnumber == 1223 else f"https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_{tmsnumber}_22_{i}.csv" for tmsnumber in tmsnumbers)
        
        traffic_df = pd.concat(map(read_csv_for_traffic, urls))
        if traffic_df is None:
            continue
        traffic_df['year'] = traffic_df['year'] + 2000
        traffic_df['date'] = compose_date(traffic_df['year'], days=traffic_df['doy'], hours=traffic_df['h'], minutes=traffic_df['m'], seconds=traffic_df['s'], milliseconds=traffic_df['ms'])
        traffic_df = traffic_df[traffic_df['faulty'] == 0]
        
        # Group by  hour
        result_df = traffic_df.groupby([pd.Grouper(key='date', freq='H'), traffic_df['class'], traffic_df['tmsnumber']]).mean()['nopeus (km/h)']
        result_df = result_df.reset_index()

        # Count the number of records per hour
        by_hour = traffic_df.groupby([pd.Grouper(key='date', freq='H'), traffic_df['class'], traffic_df['tmsnumber']]).size().reset_index(name='count')
        by_hour['speed'] = result_df['nopeus (km/h)']
        
        traffic_by_hour.append(by_hour)
    
    return pd.concat(traffic_by_hour, ignore_index=True)

stations = requests.get("https://tie.digitraffic.fi//api/tms/v1/stations")
stations_t = json.load(io.StringIO(stations.text))
weather = requests.get("https://tie.digitraffic.fi/api/weather/v1/stations")
weather_t = json.load(io.StringIO(weather.text))
gdf_weather = gpd.GeoDataFrame.from_features(weather_t["features"])
gdf = gpd.GeoDataFrame.from_features(stations_t["features"])
#ax = gdf_weather.plot()
gdf.crs = "EPSG:4326"

oulu = pd.concat([
    gdf.loc[["oulu" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["tobo testipiste" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["st815_lentokentäntie" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_rusko2" in c.lower() for c in list(gdf['name'])]],
    gdf.loc[["vt20_kiiminki" in c.lower() for c in list(gdf['name'])]]
])

oulu_stations = oulu[
    (oulu.name != "vt7_Treksilä") &
    (oulu.name != "Vt4_Oulu_Kontinkangas_Flex") &
    (oulu.name != "vt4_Oulu_Haukipudas") &
    (oulu.name != "vt4_Oulu_Kello") &
    (oulu.name != "Vt20_Oulu_Honkimaa_Flex") &
    (oulu.name != "kt45_Oulunkylä")
]

tampere = oulu_stations.to_crs(4326)

# Create hourly traffic dataframe
traffic_df_hour = create_traffic_df_by_hour(tampere)
traffic_df_hour = traffic_df_hour[traffic_df_hour['class'] != 0]

# Pivot to get counts and speeds per class in separate columns
pivot = pd.pivot_table(traffic_df_hour, values=['count', 'speed'], index=['date', 'tmsnumber'], columns='class')
# Flatten MultiIndex columns
pivot.columns = [f"{col[0]}_{col[1]}" for col in pivot.columns]
pivot = pivot.reset_index().fillna(0)

# Compute aggregated counts and weighted speeds for light and heavy vehicles
# Here we consider classes 1,6,7 as light; and classes 2,3,4,5,9 as heavy.
pivot['count_light'] = pivot[['count_1', 'count_6', 'count_7']].sum(axis=1)
pivot['count_heavy'] = pivot[['count_2', 'count_3', 'count_4', 'count_5', 'count_9']].sum(axis=1)
pivot['speed_light'] = pivot.apply(lambda row: 
                                   (row['count_1'] * row['speed_1'] + row['count_6'] * row['speed_6'] + row['count_7'] * row['speed_7']) / row['count_light']
                                   if row['count_light'] > 0 else 0, axis=1)
pivot['speed_heavy'] = pivot.apply(lambda row: 
                                   (row['count_2'] * row['speed_2'] + row['count_3'] * row['speed_3'] + row['count_4'] * row['speed_4'] + 
                                    row['count_5'] * row['speed_5'] + row['count_9'] * row['speed_9']) / row['count_heavy']
                                   if row['count_heavy'] > 0 else 0, axis=1)

# Assign each row to a day or night period.
# Day: hour between 07:00 (inclusive) and 22:00 (exclusive), otherwise Night.
pivot['period'] = pivot['date'].dt.hour.apply(lambda h: 'd' if 7 <= h < 22 else 'n')

# List of classes to aggregate
classes = [1,2,3,4,5,6,7,9]
# List for light and heavy aggregates.
light_cols = ('count_1','count_6','count_7')
heavy_cols = ('count_2','count_3','count_4','count_5','count_9')

# Function to aggregate a group (per tmsnumber and period)
def aggregate_group(group):
    agg = {}
    # For each vehicle class, sum counts and compute weighted average speed.
    for cls in classes:
        cnt_col = f"count_{cls}"
        spd_col = f"speed_{cls}"
        total_count = group[cnt_col].sum()
        agg[cnt_col] = total_count
        # Weighted average speed; if total_count is zero, then 0.
        if total_count > 0:
            weighted_speed = (group[cnt_col] * group[spd_col]).sum() / total_count
        else:
            weighted_speed = 0
        agg[spd_col] = weighted_speed
    # Aggregated light vehicles
    total_light = group[list(light_cols)].sum().sum()
    agg['count_light'] = total_light
    if total_light > 0:
        weighted_speed_light = (group['count_1'] * group['speed_1'] + 
                                group['count_6'] * group['speed_6'] + 
                                group['count_7'] * group['speed_7']).sum() / total_light
    else:
        weighted_speed_light = 0
    agg['speed_light'] = weighted_speed_light

    # Aggregated heavy vehicles
    total_heavy = group[list(heavy_cols)].sum().sum()
    agg['count_heavy'] = total_heavy
    if total_heavy > 0:
        weighted_speed_heavy = (group['count_2'] * group['speed_2'] + 
                                group['count_3'] * group['speed_3'] +
                                group['count_4'] * group['speed_4'] +
                                group['count_5'] * group['speed_5'] +
                                group['count_9'] * group['speed_9']).sum() / total_heavy
    else:
        weighted_speed_heavy = 0
    agg['speed_heavy'] = weighted_speed_heavy

    return pd.Series(agg)


# Group by tmsnumber and period to merge hourly data into day and night aggregates.
grouped = pivot.groupby(['tmsnumber', 'period']).apply(aggregate_group).reset_index()

# Pivot the groups so that day (d) and night (n) aggregates become separate columns.
merged = grouped.pivot(index='tmsnumber', columns='period')
# Flatten the columns: e.g., ("count_1", "d") becomes "d_count_1"
merged.columns = [f"{col[1]}_{col[0]}" for col in merged.columns]
merged = merged.reset_index()

# Rename column to "tmsNumber"
merged.rename(columns={'tmsnumber': 'tmsNumber'}, inplace=True)

# Add date column: For tmsNumber==1223, date is "10.09.2021", else "09.09.2022".
merged['date'] = merged['tmsNumber'].apply(lambda x: "10.09.2021" if x == 1223 else "09.09.2022")

# Reorder columns as specified in the output.
# Desired order:
columns_order = ['date', 'tmsNumber',
    'd_count_1','d_count_2','d_count_3','d_count_4','d_count_5','d_count_6','d_count_7','d_count_9',
    'd_speed_1','d_speed_2','d_speed_3','d_speed_4','d_speed_5','d_speed_6','d_speed_7','d_speed_9',
    'd_count_light','d_count_heavy','d_speed_light','d_speed_heavy',
    'n_count_1','n_count_2','n_count_3','n_count_4','n_count_5','n_count_6','n_count_7','n_count_9',
    'n_speed_1','n_speed_2','n_speed_3','n_speed_4','n_speed_5','n_speed_6','n_speed_7','n_speed_9',
    'n_count_light','n_count_heavy','n_speed_light','n_speed_heavy'
]
# Some columns might be missing if there were no counts in a given period – fill missing columns with 0.
for col in columns_order:
    if col not in merged.columns:
        merged[col] = 0

merged = merged[columns_order]

# Save the result to CSV.
merged.to_csv('files_QGIS/oulu_traffic2.csv', index=False)

## Check availability

In [1]:
import requests

tms_range = [21201]



# Check all days in 2022
# i = 22
i_range = range(21, 25)  # Year (2022)
j_range = range(1, 366)  # Day of year (1 to 365)
j = 252

# test the URL
with requests.Session() as session:
    for tms_number in tms_range:
        for i in i_range:
            url = f"https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_{tms_number}_{i}_{j}.csv"
            
            try:
                # GET request
                response = session.get(url)
                
                # Check if working
                if response.status_code == 200:
                    print(f"URL found: {url}")
            except requests.RequestException:
                pass

URL found: https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_21201_22_252.csv
URL found: https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_21201_23_252.csv
URL found: https://tie.digitraffic.fi/api/tms/v1/history/raw/lamraw_21201_24_252.csv


In [ ]:
1238 == 21206
1244 == 21205